In [1]:
import numpy as np
import pandas as pd
import os
import glob
import re
import hashlib
from pathlib import Path

In [2]:
years=["2017","2018","2019","2020","2021","2022","2023","2024","2025"]
#years=["2022","2023","2024","2025"]
objective="summary"

In [3]:
ruta_archivos_csv = r"C:\Users\ruben\Documents\Universidad\TEG\Codigos\VSCode"
for year in years:
    archivos_csv_summary = [os.path.join(ruta_archivos_csv, archivo) for archivo in os.listdir(ruta_archivos_csv) if archivo.endswith(".csv") and "summary" in archivo and year in archivo]
    print("Cantidad de archivos summary que se cargarán en ", year, " : ", len(archivos_csv_summary))

Cantidad de archivos summary que se cargarán en  2017  :  5358
Cantidad de archivos summary que se cargarán en  2018  :  11680
Cantidad de archivos summary que se cargarán en  2019  :  12042
Cantidad de archivos summary que se cargarán en  2020  :  10678
Cantidad de archivos summary que se cargarán en  2021  :  13890
Cantidad de archivos summary que se cargarán en  2022  :  13476
Cantidad de archivos summary que se cargarán en  2023  :  14026
Cantidad de archivos summary que se cargarán en  2024  :  13972
Cantidad de archivos summary que se cargarán en  2025  :  13644


In [4]:
# Reacomodamos los nombres

mapa_columnas = {
    "Team_id":["team"],
    "Player":["Unnamed: 0_level_0 Player"],
    "Number":["Unnamed: 1_level_0 #"],
    "Country":["Unnamed: 2_level_0 Nation"],
    "Position":["Unnamed: 3_level_0 Pos"],
    "Age":["Unnamed:4_level_0 Age"],
    "Min":["Unnamed: 5_level_0 Min"],
    "Goals":["Performance Gls"],
    "Assists":["Performance Ast"],
    "GoalsPen":["Performance PK"],
    "AttPen":["Performance PKatt"],
    "Shots":["Performance Sh"],
    "ShTarget":["Performance SoT"],
    "YellowCard":["Performance CrdY"],
    "RedCard":["Performance CrdR"],
    "Fouls":["Performance Fls"],
    "FoulsD":["Performance Fld"],
    "Offside":["Performance Off"],
    "Crosses":["Performance Crs"],
    "TacklesW":["Performance TklW"],
    "Interceptions":["Performance Int"],
    "OwnGoals":["Performance OG"],
    "PenKickGen":["Performance PKwon"],
    "PenKickCon":["Performance PKcon"]
}

In [5]:
print(pd.read_csv(archivos_csv_summary[1]).head())
test=pd.read_csv(archivos_csv_summary[1]).head()

   _table_index               _table_id                  _table_name  \
0             1  stats_06c1606c_summary  Banfield Player Stats Table   
1             1  stats_06c1606c_summary  Banfield Player Stats Table   
2             1  stats_06c1606c_summary  Banfield Player Stats Table   
3             1  stats_06c1606c_summary  Banfield Player Stats Table   
4             1  stats_06c1606c_summary  Banfield Player Stats Table   

                                            match_id        date  \
0  Barracas-Central-Banfield-February-1-2025-Liga...  2025-02-01   
1  Barracas-Central-Banfield-February-1-2025-Liga...  2025-02-01   
2  Barracas-Central-Banfield-February-1-2025-Liga...  2025-02-01   
3  Barracas-Central-Banfield-February-1-2025-Liga...  2025-02-01   
4  Barracas-Central-Banfield-February-1-2025-Liga...  2025-02-01   

                                league      team     kind  \
0  ar Liga Profesional Argentina Table  06c1606c  summary   
1  ar Liga Profesional Argentina Tab

In [6]:
print(sorted(test.columns))

['Performance Ast', 'Performance CrdR', 'Performance CrdY', 'Performance Crs', 'Performance Fld', 'Performance Fls', 'Performance Gls', 'Performance Int', 'Performance OG', 'Performance Off', 'Performance PK', 'Performance PKatt', 'Performance PKcon', 'Performance PKwon', 'Performance Sh', 'Performance SoT', 'Performance TklW', 'Unnamed: 0_level_0 Player', 'Unnamed: 1_level_0 #', 'Unnamed: 2_level_0 Nation', 'Unnamed: 3_level_0 Pos', 'Unnamed: 4_level_0 Age', 'Unnamed: 5_level_0 Min', '_table_id', '_table_index', '_table_name', 'date', 'kind', 'league', 'match_id', 'team']


In [7]:
# Recolectar todos los CSV de summary por anio
archivos_csv_summary_all = []
for year in years:
    archivos_csv_summary_all.extend([
        os.path.join(ruta_archivos_csv, archivo)
        for archivo in os.listdir(ruta_archivos_csv)
        if archivo.endswith(".csv") and "summary" in archivo and year in archivo
    ])
print("Total archivos summary:", len(archivos_csv_summary_all))

def _normalize_team_name(raw):
    if raw is None:
        return None
    raw = raw.strip("_")
    raw = re.sub(r"^[a-z]{2,3}_", "", raw)
    raw = re.sub(r"_[a-z]{2,3}$", "", raw)
    name = raw.replace("_", " ")
    name = re.sub(r"\s+", " ", name).strip()
    return name

def parse_meta_from_filename(path_csv):
    name = os.path.basename(path_csv)
    base = re.sub(r"_summary\.csv$", "", name)
    if "__" not in base:
        return {"home_team": None, "away_team": None, "Team": None, "match_id": None}
    left, right = base.rsplit("__", 1)
    m_side = re.match(r"0(\d+?)_(.+)$", right)
    if not m_side:
        return {"home_team": None, "away_team": None, "Team": None, "match_id": None}
    side_raw = m_side.group(1)
    match_id = m_side.group(2)
    m_date = re.search(r"(\d{2}-\d{2}-\d{4})", left)
    if not m_date:
        return {"home_team": None, "away_team": None, "Team": None, "match_id": match_id}
    after = left[m_date.end() + 1 :] if m_date.end() + 1 < len(left) else ""
    partes = re.split(r"-vs(?:\._|_)", after, maxsplit=1)
    if len(partes) != 2:
        partes = re.split(r"-vs-", after, maxsplit=1)
    if len(partes) != 2:
        partes = re.split(r"vs(?:\._|_)", after, maxsplit=1)
    if len(partes) != 2:
        return {"home_team": None, "away_team": None, "Team": None, "match_id": match_id}
    home_raw, away_raw = partes
    home_team = _normalize_team_name(home_raw)
    away_team = _normalize_team_name(away_raw)
    # team = home_team if side_raw == "00" else away_team
    team = home_team if side_raw == "0" else (away_team if side_raw in ["1", "6"] else None)
    return {
        "home_team": home_team,
        "away_team": away_team,
        "Team": team,
        "match_id": match_id,
    }

# Cargar CSV y agregar columnas desde el nombre del archivo
#USE_SAMPLE = True
#SAMPLE_SIZE = 50
#PROGRESS_EVERY = 1000

#if USE_SAMPLE:
    #np.random.seed(25)
    #total_archivos = len(archivos_csv_summary_all)
    #sample_size = min(SAMPLE_SIZE, total_archivos)
    #archivos_a_procesar = list(np.random.choice(archivos_csv_summary_all, size=sample_size, replace=False))
    #print("Procesando muestra:", len(archivos_a_procesar))
#else:
    #archivos_a_procesar = archivos_csv_summary_all
    #print("Procesando todos los archivos:", len(archivos_a_procesar))

#dfs_raw = []
#total = len(archivos_a_procesar)
#for i, path_csv in enumerate(archivos_a_procesar, 1):
    #df = pd.read_csv(path_csv)
    #meta = parse_meta_from_filename(path_csv)
    #df["home_team"] = meta["home_team"]
    #df["away_team"] = meta["away_team"]
    #df["Team"] = meta["Team"]
    #df["match_id"] = meta.get("match_id")
    #df["source_file"] = os.path.basename(path_csv)
    #dfs_raw.append(df)
    #if i % PROGRESS_EVERY == 0 or i == total:
        #print(f"Procesados {i}/{total}")

#df_raw_all = pd.concat(dfs_raw, ignore_index=True)
#df_raw_all.head()

Total archivos summary: 108766


In [8]:
def read_last_row(path_csv):
    try:
        df_tail = pd.read_csv(path_csv).tail(1)
    except UnicodeDecodeError:
        df_tail = pd.read_csv(path_csv, encoding="latin-1").tail(1)
    return df_tail

In [9]:
# Usar solo la ultima fila (totales), limpiar columnas y agregar metadatos
def read_last_row(path_csv):
    try:
        df_tail = pd.read_csv(path_csv).tail(1)
    except UnicodeDecodeError:
        df_tail = pd.read_csv(path_csv, encoding="latin-1").tail(1)
    return df_tail

def rename_columns(df):
    rename_map = {}
    for new_name, old_names in mapa_columnas.items():
        for old_name in old_names:
            if old_name in df.columns:
                rename_map[old_name] = new_name
    return df.rename(columns=rename_map)

drop_cols = {
    "Unnamed: 0_level_0 Player",
    "Unnamed: 1_level_0 #",
    "Unnamed: 2_level_0 Nation",
    "Unnamed: 3_level_0 Pos",
    "Unnamed:4_level_0 Age",
    "Unnamed: 4_level_0 Age",
    "Unnamed: 5_level_0 Min",
    "Carries Carries",
    "Carries PrgC",
    "Expected npxG",
    "Expected xAG",
    "Expected xG",
    "Passes Att",
    "Passes Cmp",
    "Passes Cmp%",
    "Passes PrgP",
    "Performance Blocks",
    "Performance Touches",
    "SCA GCA",
    "SCA SCA",
    "Take-Ons Att",
    "Take-Ons Succ",
}

meta_cols = ["home_team", "away_team", "Team", "match_id", "source_file"]
desired_cols = list(mapa_columnas.keys())
all_cols = desired_cols + meta_cols

rows_totales = []
total_archivos = len(archivos_csv_summary_all)
for path_csv in archivos_csv_summary_all:
    df_last = read_last_row(path_csv)
    df_last = df_last.drop(columns=[c for c in drop_cols if c in df_last.columns], errors="ignore")
    df_last = rename_columns(df_last)
    # Evitar columnas duplicadas antes de reindexar
    df_last = df_last.loc[:, ~df_last.columns.duplicated()]
    meta = parse_meta_from_filename(path_csv)
    df_last["home_team"] = meta["home_team"]
    df_last["away_team"] = meta["away_team"]
    df_last["Team"] = meta["Team"]
    df_last["match_id"] = meta.get("match_id")
    df_last["source_file"] = os.path.basename(path_csv)
    df_last = df_last.reindex(columns=all_cols)
    rows_totales.append(df_last)

df_totales = pd.concat(rows_totales, ignore_index=True)
print("Archivos procesados:", total_archivos)
print("Columnas finales:", df_totales.columns.tolist())
df_totales.head()

KeyboardInterrupt: 

In [ ]:
# Prueba rapida con una muestra pequena
meta_cols = ["home_team", "away_team", "Team", "match_id", "source_file"]
sample_size = 50
np.random.seed(18)
total_archivos = len(archivos_csv_summary_all)
sample_size = min(sample_size, total_archivos)
sample_files = list(np.random.choice(archivos_csv_summary_all, size=sample_size, replace=False))
rows_sample = []
columnas_sample = set()
for path_csv in sample_files:
    df_last = read_last_row(path_csv)
    meta = parse_meta_from_filename(path_csv)
    df_last["home_team"] = meta["home_team"]
    df_last["away_team"] = meta["away_team"]
    df_last["Team"] = meta["Team"]
    df_last["match_id"] = meta.get("match_id")
    df_last["source_file"] = os.path.basename(path_csv)
    columnas_sample.update([c for c in df_last.columns if c not in meta_cols])
    rows_sample.append(df_last)

df_sample = pd.concat(rows_sample, ignore_index=True)
print("Archivos en muestra:", len(sample_files))
print("Columnas detectadas (sin meta):", len(columnas_sample))
print(sorted(columnas_sample))
print("Columnas meta:", meta_cols)
df_sample[meta_cols].head(50)

Archivos en muestra: 50
Columnas detectadas (sin meta): 30
['Performance Ast', 'Performance CrdR', 'Performance CrdY', 'Performance Crs', 'Performance Fld', 'Performance Fls', 'Performance Gls', 'Performance Int', 'Performance OG', 'Performance Off', 'Performance PK', 'Performance PKatt', 'Performance PKcon', 'Performance PKwon', 'Performance Sh', 'Performance SoT', 'Performance TklW', 'Unnamed: 0_level_0 Player', 'Unnamed: 1_level_0 #', 'Unnamed: 2_level_0 Nation', 'Unnamed: 3_level_0 Pos', 'Unnamed: 4_level_0 Age', 'Unnamed: 5_level_0 Min', '_table_id', '_table_index', '_table_name', 'date', 'kind', 'league', 'team']
Columnas meta: ['home_team', 'away_team', 'Team', 'match_id', 'source_file']


,home_team,away_team,Team,match_id,source_file
0,Hertha BSC,Nurnberg,Nurnberg,6f2c108c,players_de_Bundesliga-25-08-2018-Hertha_BSC-vs...
1,Juventude,America–MG,Juventude,d081b697,players_br_Série_A-09-10-2021-Juventude-vs_Ame...
2,Atle Tucuman,Ind. Rivadavia,Ind. Rivadavia,8a9d5afa,players_ar_Liga_Profesional_Argentina-04-08-20...
3,FC Juarez,Atlas,FC Juarez,29bff345,players_mx_Liga_MX-05-07-2024-FC_Juarez-vs_Atl...
4,Liverpool,Southampton,Liverpool,822bd0ba,players_eng_Premier_League-18-11-2017-Liverpoo...
5,RB Leipzig,Wolfsburg,RB Leipzig,acbb6a5b,players_de_Bundesliga-23-01-2022-RB_Leipzig-vs...
6,Marseille,Bordeaux,Bordeaux,123f3efe,players_fr_Ligue_1-18-02-2018-Marseille-vs_Bor...
7,Kyoto Sanga,Sagan Tosu,Kyoto Sanga,e5ce7354,players_jp_J1_League-23-04-2023-Kyoto_Sanga-vs...
8,Al-Taawoun,Al-Nassr,Al-Nassr,6baef27f,players_sa_Saudi_Pro_League-17-01-2025-Al-Taaw...
9,Estudiantes Merida,Palestino,Palestino,fccf2756,players_Copa_Sudamericana-25-05-2023-Estudiant...


In [ ]:
# Debug: archivos con meta vacia
fallos_meta = []
for path_csv in sample_files:
    meta = parse_meta_from_filename(path_csv)
    if not meta.get("home_team") or not meta.get("away_team") or not meta.get("Team"):
        fallos_meta.append((os.path.basename(path_csv), meta))

print("Archivos con meta incompleta:", len(fallos_meta))
for name, meta in fallos_meta[:10]:
    print(name, "->", meta)

Archivos con meta incompleta: 0


In [ ]:
# Exportar df_totales como CSV crudo
output_path_raw = os.path.join(ruta_archivos_csv, "df_raw.csv")
df_totales.to_csv(output_path_raw, index=False)
print("CSV exportado:", output_path_raw)

CSV exportado: C:\Users\ruben\Documents\Universidad\TEG\Codigos\VSCode\df_raw.csv


In [ ]:
# Limpiar columnas finales y exportar CSV crudo
cols_to_drop_final = ["Player", "Number", "Country", "Position", "Age", "Min"]
df_totales2 = df_totales.drop(columns=cols_to_drop_final, errors="ignore")
output_path_raw2 = os.path.join(ruta_archivos_csv, "df_raw_limpio.csv")
df_totales2.to_csv(output_path_raw2, index=False)
print("CSV exportado:", output_path_raw2)

CSV exportado: C:\Users\ruben\Documents\Universidad\TEG\Codigos\VSCode\df_raw_limpio.csv


In [2]:
# ------------------------------------------------------------------
# Montaje de Google Drive (ejecutar solo si estás en Colab)
# ------------------------------------------------------------------
from google.colab import drive
drive.mount('/content/drive')

# ------------------------------------------------------------------
# CONFIGURACIÓN GLOBAL
# ------------------------------------------------------------------
BASE_DATA_DIR = "/content/drive/MyDrive/TEG02-2025Arellano/Data"
ruta_totales = os.path.join(BASE_DATA_DIR, "df_raw.csv")
df_totales=pd.read_csv(ruta_totales)
ruta_totales2 = os.path.join(BASE_DATA_DIR, "df_raw_limpio.csv")
df_totales2=pd.read_csv(ruta_totales2)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [3]:
cols_to_drop_final = ["Player", "Number", "Country", "Position", "Age", "Min"]
df_totales2 = df_totales.drop(columns=cols_to_drop_final, errors="ignore")

In [4]:
import re

def parse_source_filename(filename):
    """
    Extrae: torneo, fecha, home_team, away_team, side (0/1), team_id, match_key.
    Soporta nombres de torneo con guiones y equipos con guiones.
    """
    pattern = r'^players_(.+)-(\d{2}-\d{2}-\d{4})-(.+?)-vs[_-](.+?)__(0[01])_([a-f0-9]+)_summary\.csv$'
    m = re.match(pattern, filename)
    if not m:
        return None
    torneo, date_str, home, away, side, team_id = m.groups()
    return {
        'torneo': torneo,
        'date': date_str,
        'home_team': home,
        'away_team': away,
        'side': int(side),
        'team_id': team_id,
        'match_key': f"{date_str}_{home}_{away}"   # identificador único del partido
    }

# Aplicar a todas las filas
parsed = df_totales2['source_file'].apply(parse_source_filename)
valid_mask = parsed.notna()
df_valid = df_totales2[valid_mask].copy()
df_valid['parsed'] = parsed[valid_mask]

# Extraer columnas auxiliares
df_valid['date'] = df_valid['parsed'].apply(lambda x: x['date'])
df_valid['home_team_parsed'] = df_valid['parsed'].apply(lambda x: x['home_team'])
df_valid['away_team_parsed'] = df_valid['parsed'].apply(lambda x: x['away_team'])
df_valid['side'] = df_valid['parsed'].apply(lambda x: x['side'])
df_valid['team_id_parsed'] = df_valid['parsed'].apply(lambda x: x['team_id'])
df_valid['match_key'] = df_valid['parsed'].apply(lambda x: x['match_key'])

# Obtener home_id y away_id para cada match_key
match_ids = {}
for key, group in df_valid.groupby('match_key'):
    home_ids = group[group['side'] == 0]['team_id_parsed'].unique()
    away_ids = group[group['side'] == 1]['team_id_parsed'].unique()
    match_ids[key] = (home_ids[0] if len(home_ids) == 1 else None,
                      away_ids[0] if len(away_ids) == 1 else None)

df_valid['home_team_id_from_name'] = df_valid['match_key'].apply(lambda k: match_ids.get(k, (None, None))[0])
df_valid['away_team_id_from_name'] = df_valid['match_key'].apply(lambda k: match_ids.get(k, (None, None))[1])

# Construir match_id_new (YYYYMMDD_homeid_awayid) solo si ambos IDs existen
df_valid['match_id_new'] = df_valid.apply(
    lambda row: (
        f"{row['date'].split('-')[2]}{row['date'].split('-')[1]}{row['date'].split('-')[0]}_{row['home_team_id_from_name']}_{row['away_team_id_from_name']}"
    )
    if row['home_team_id_from_name'] is not None and row['away_team_id_from_name'] is not None
    else None,
    axis=1
)
# Quedarse solo con las filas que tienen match_id_new completo
df_totales2 = df_valid.drop(columns=['parsed', 'date', 'home_team_parsed', 'away_team_parsed',
                                  'side', 'team_id_parsed', 'match_key',
                                  'home_team_id_from_name', 'away_team_id_from_name'])

print(f"Filas originales: {len(df_totales2)}")
print(f"Filas con match_id_new completo: {len(df_valid)}")
print("Muestra de match_id_new:")
print(df_totales2[['source_file', 'match_id_new']].head(10))

# Extraer match_date desde source_file
df_totales2['match_date'] = df_totales2['source_file'].str.extract(r'(\d{2}-\d{2}-\d{4})', expand=False)

# Extraer team_id_home y team_id_away desde match_id_new (formato: YYYYMMDD_homeid_awayid)
# Si match_id_new tiene exactamente 3 partes separadas por '_'
split_ids = df_totales2['match_id_new'].str.split('_', expand=True)
# split_ids[0] es la fecha, split_ids[1] es home, split_ids[2] es away
df_totales2['team_id_home'] = split_ids[1]
df_totales2['team_id_away'] = split_ids[2]

# Opcional: si quieres que sean strings (ya lo son)

Filas originales: 98972
Filas con match_id_new completo: 98972
Muestra de match_id_new:
                                         source_file  \
0  players_ar_Superliga_Argentina-01-10-2017-Boca...   
1  players_ar_Superliga_Argentina-01-10-2017-Boca...   
2  players_ar_Superliga_Argentina-01-10-2017-Inde...   
3  players_ar_Superliga_Argentina-01-10-2017-Inde...   
4  players_ar_Superliga_Argentina-01-10-2017-San_...   
5  players_ar_Superliga_Argentina-01-10-2017-San_...   
6  players_ar_Superliga_Argentina-01-10-2017-Tall...   
7  players_ar_Superliga_Argentina-01-10-2017-Tall...   
8  players_ar_Superliga_Argentina-01-10-2017-Tigr...   
9  players_ar_Superliga_Argentina-01-10-2017-Tigr...   

                 match_id_new  
0  20171001_795ca75e_aa68918c  
1  20171001_795ca75e_aa68918c  
2  20171001_40bb0ce9_41c139b6  
3  20171001_40bb0ce9_41c139b6  
4  20171001_d7a2603d_87a920fa  
5  20171001_d7a2603d_87a920fa  
6  20171001_ceda2145_0e6891d5  
7  20171001_ceda2145_0e6891d5  
8  2017

In [5]:
# Crear team_id_home, team_id_away y match_id_new
# Deprecada en favor de la anterior
#df_totales2 = df_totales2.copy()
#raw_date = df_totales2["source_file"].str.extract(r"(\d{2}-\d{2}-\d{4})", expand=False)
#df_totales2["match_date"] = raw_date
#match_date_jp = raw_date.str.replace(r"(\d{2})-(\d{2})-(\d{4})", r"\3\2\1", regex=True)
#team_id_map = (
    #df_totales2[["Team", "Team_id"]]
    #.dropna()
    #.drop_duplicates("Team")
    #.set_index("Team")["Team_id"]
#)
#df_totales2["team_id_home"] = df_totales2["home_team"].map(team_id_map)
#df_totales2["team_id_away"] = df_totales2["away_team"].map(team_id_map)
#df_totales2["match_id_new"] = (
    #match_date_jp.fillna("") + "_"+
    #df_totales2["team_id_home"].fillna("").astype(str) + "_"+
    #df_totales2["team_id_away"].fillna("").astype(str)
#)
#df_totales2[["match_id", "match_id_new", "match_date", "team_id_home", "team_id_away"]].head(50)

In [6]:
#cols_to_drop_final = ["Player", "Number", "Country", "Position", "Age", "Min"]
#df_totales2 = df_totales.drop(columns=cols_to_drop_final, errors="ignore")

In [7]:
# Revisar pares consecutivos de match_id en df_totales2
df_check = df_totales2.reset_index(drop=True)
mismatched_pairs = []
for i in range(0, len(df_check) - 1, 2):
    if df_check.loc[i, "match_id_new"] != df_check.loc[i + 1, "match_id_new"]:
        mismatched_pairs.append((i, df_check.loc[i, "match_id_new"], df_check.loc[i + 1, "match_id_new"]))

print("Pares con match_id distinto:", len(mismatched_pairs))
if mismatched_pairs:
    print("Ejemplos:")
    for row in mismatched_pairs[:10]:
        print(row)

# Revisar match_id con cantidad impar (posibles filas solitarias)
counts = df_check["match_id_new"].value_counts(dropna=False)
odd_match_ids = counts[counts % 2 == 1]
print("match_id con cantidad impar:", len(odd_match_ids))
odd_match_ids.head(10)

Pares con match_id distinto: 0
match_id con cantidad impar: 0


,count
match_id_new,


In [8]:
# Encontrar match_id solitarios o con mas de 2 filas
counts = df_totales2["match_id_new"].value_counts(dropna=False)
anomalos = counts[(counts == 1) | (counts > 2)]
print("match_id anomalo (1 o >2):", len(anomalos))
anomalos.head(20)

match_id anomalo (1 o >2): 2788


,count
match_id_new,
20241019_a8535c0e_d254c5db,4
20241019_2a81b275_dcbc2e42,4
20240320_2583cf18_f4da11f5,4
20240320_7701ed02_a2c7af20,4
20240320_2a81b275_67eb6cd0,4
20220306_4a2c27a3_6507d2b8,4
20240320_bcdc5828_a8535c0e,4
20240420_0f8ef17f_7701ed02,4
20220714_9aa97c75_0e6891d5,4


In [9]:
# Mostrar filas de los match_id_new anómalos
anom_ids = anomalos.index.tolist()
cols_show = [
    "match_id_new",
    "match_date",
    "home_team",
    "away_team",
    "Team",
    "Team_id",
    "team_id_home",
    "team_id_away",
    "source_file",
]
df_anom = (
    df_totales2[df_totales2["match_id_new"].isin(anom_ids)]
    .sort_values(["match_id_new", "Team"])
    .reset_index(drop=True)
 )
df_anom[cols_show].head()

,match_id_new,match_date,home_team,away_team,Team,Team_id,team_id_home,team_id_away,source_file
0,20170805_33e6936d_3ded797c,05-08-2017,Albirex Niigata,Yokohama F. Marinos,Albirex Niigata,33e6936d,33e6936d,3ded797c,players_jp_J1_League-05-08-2017-Albirex_Niigat...
1,20170805_33e6936d_3ded797c,05-08-2017,Albirex Niigata,Yokohama F Marinos,Albirex Niigata,33e6936d,33e6936d,3ded797c,players_jp_J1_League-05-08-2017-Albirex_Niigat...
2,20170805_33e6936d_3ded797c,05-08-2017,Albirex Niigata,Yokohama F Marinos,Yokohama F Marinos,3ded797c,33e6936d,3ded797c,players_jp_J1_League-05-08-2017-Albirex_Niigat...
3,20170805_33e6936d_3ded797c,05-08-2017,Albirex Niigata,Yokohama F. Marinos,Yokohama F. Marinos,3ded797c,33e6936d,3ded797c,players_jp_J1_League-05-08-2017-Albirex_Niigat...
4,20170806_7701ed02_48ab0f5f,06-08-2017,Deportivo Tachira,Atletico Venezuela,Atletico Venezuela,48ab0f5f,7701ed02,48ab0f5f,players_ve_Venezuelan_Primera_Division-06-08-2...


In [10]:
# Detectar archivos casi duplicados por normalizacion del nombre
def normalize_source_name(name):
    if pd.isna(name):
        return name
    base = name.lower()
    base = re.sub(r"[^a-z0-9]+", "_", base)
    base = re.sub(r"_+", "_", base).strip("_")
    return base
df_anom = df_anom.copy()
df_anom["source_key"] = df_anom["source_file"].apply(normalize_source_name)
dupe_keys = df_anom["source_key"].value_counts()
dupe_keys = dupe_keys[dupe_keys > 1]
print("Grupos con nombres casi duplicados:", len(dupe_keys))
if len(dupe_keys) > 0:
    cols_data = [c for c in df_anom.columns if c not in ["source_file", "source_key"]]
    df_dupes = df_anom[df_anom["source_key"].isin(dupe_keys.index)].copy()
    df_dupes["non_nulls"] = df_dupes[cols_data].notna().sum(axis=1)
    df_dupes = df_dupes.sort_values(["source_key", "non_nulls"], ascending=[True, False])
    df_dupes[["source_key", "source_file", "non_nulls", "Team", "Team_id", "home_team", "away_team", "match_date"]].head(200)

Grupos con nombres casi duplicados: 1820


In [11]:
# Limpiar NaN en match_date y quedarnos con el archivo mas completo por source_key
def normalize_source_name(name):
    if pd.isna(name):
        return name
    base = name.lower()
    base = re.sub(r"[^a-z0-9]+", "_", base)
    base = re.sub(r"_+", "_", base).strip("_")
    return base
df_clean = df_totales2.dropna(subset=["match_date"]).copy()
df_clean["source_key"] = df_clean["source_file"].apply(normalize_source_name)
cols_data = [c for c in df_clean.columns if c not in ["source_file", "source_key"]]
df_clean["non_nulls"] = df_clean[cols_data].notna().sum(axis=1)
df_totales2_dedup = (
    df_clean.sort_values(["source_key", "non_nulls"], ascending=[True, False])
    .drop_duplicates(subset=["source_key"], keep="first")
    .drop(columns=["non_nulls"])
    .reset_index(drop=True)
)
print("Filas originales:", len(df_totales2))
print("Filas sin match_date:", len(df_clean))
print("Filas deduplicadas:", len(df_totales2_dedup))
df_totales2_dedup[["source_key", "source_file", "match_date", "Team", "Team_id"]].head(20)

Filas originales: 98972
Filas sin match_date: 98972
Filas deduplicadas: 97152


,source_key,source_file,match_date,Team,Team_id
0,players_ar_liga_profesional_argentina_01_02_20...,players_ar_Liga_Profesional_Argentina-01-02-20...,01-02-2025,Barracas Central,1d89634b
1,players_ar_liga_profesional_argentina_01_02_20...,players_ar_Liga_Profesional_Argentina-01-02-20...,01-02-2025,Banfield,06c1606c
2,players_ar_liga_profesional_argentina_01_02_20...,players_ar_Liga_Profesional_Argentina-01-02-20...,01-02-2025,Lanus,11b6dba8
3,players_ar_liga_profesional_argentina_01_02_20...,players_ar_Liga_Profesional_Argentina-01-02-20...,01-02-2025,Sarmiento–J,e9ae80b7
4,players_ar_liga_profesional_argentina_01_02_20...,players_ar_Liga_Profesional_Argentina-01-02-20...,01-02-2025,San Martin–SJ,d7a2603d
5,players_ar_liga_profesional_argentina_01_02_20...,players_ar_Liga_Profesional_Argentina-01-02-20...,01-02-2025,Rosario Central,87a920fa
6,players_ar_liga_profesional_argentina_01_03_20...,players_ar_Liga_Profesional_Argentina-01-03-20...,01-03-2025,Aldosivi,e5927bd8
7,players_ar_liga_profesional_argentina_01_03_20...,players_ar_Liga_Profesional_Argentina-01-03-20...,01-03-2025,Sarmiento–J,e9ae80b7
8,players_ar_liga_profesional_argentina_01_03_20...,players_ar_Liga_Profesional_Argentina-01-03-20...,01-03-2025,River Plate,ef99c78c
9,players_ar_liga_profesional_argentina_01_03_20...,players_ar_Liga_Profesional_Argentina-01-03-20...,01-03-2025,Estudiantes–LP,df734df9


In [12]:
# Revisar pares y anomalos en df_totales2_dedup
df_check_dedup = df_totales2_dedup.reset_index(drop=True)
mismatched_pairs_dedup = []
for i in range(0, len(df_check_dedup) - 1, 2):
    if df_check_dedup.loc[i, "match_id_new"] != df_check_dedup.loc[i + 1, "match_id_new"]:
        mismatched_pairs_dedup.append(
            (i, df_check_dedup.loc[i, "match_id_new"], df_check_dedup.loc[i + 1, "match_id_new"])
        )
print("Pares con match_id_new distinto (dedup):", len(mismatched_pairs_dedup))
if mismatched_pairs_dedup:
    print("Ejemplos:")
    for row in mismatched_pairs_dedup[:10]:
        print(row)
counts_dedup = df_check_dedup["match_id_new"].value_counts(dropna=False)
anomalos_dedup = counts_dedup[(counts_dedup == 1) | (counts_dedup > 2)]
print("match_id_new anomalo (1 o >2) en dedup:", len(anomalos_dedup))
anomalos_dedup.head(20)

Pares con match_id_new distinto (dedup): 0
match_id_new anomalo (1 o >2) en dedup: 1878


,count
match_id_new,
20171018_48ab0f5f_3d2f4487,4
20220918_bcdc5828_67eb6cd0,4
20220918_22460fcd_a8535c0e,4
20220918_a2c7af20_3f2d6531,4
20220918_6dd8415f_0f8ef17f,4
20220918_7701ed02_dcbc2e42,4
20220918_6236a52b_3d2f4487,4
20220918_90e07850_d254c5db,4
20201004_a8535c0e_a2c7af20,4


In [13]:
# Mostrar filas anómalas en df_totales2_dedup
anom_ids_dedup = anomalos_dedup.index.tolist()
cols_show_dedup = [
    "match_id_new",
    "match_date",
    "home_team",
    "away_team",
    "Team",
    "Team_id",
    "team_id_home",
    "team_id_away",
    "source_file",
    "source_key",
]
df_anom_dedup = (
    df_totales2_dedup[df_totales2_dedup["match_id_new"].isin(anom_ids_dedup)]
    .sort_values(["match_id_new", "Team"])
    .reset_index(drop=True)
 )
df_anom_dedup[cols_show_dedup].head(200)

,match_id_new,match_date,home_team,away_team,Team,Team_id,team_id_home,team_id_away,source_file,source_key
0,20170806_7701ed02_48ab0f5f,06-08-2017,Deportivo Tachira,Atletico Venezuela,Atletico Venezuela,48ab0f5f,7701ed02,48ab0f5f,players_ve_Venezuelan_Primera_División-06-08-2...,players_ve_venezuelan_primera_divisi_n_06_08_2...
1,20170806_7701ed02_48ab0f5f,06-08-2017,Deportivo Tachira,Atletico Venezuela,Atletico Venezuela,48ab0f5f,7701ed02,48ab0f5f,players_ve_Venezuelan_Primera_Division-06-08-2...,players_ve_venezuelan_primera_division_06_08_2...
2,20170806_7701ed02_48ab0f5f,06-08-2017,Deportivo Tachira,Atletico Venezuela,Deportivo Tachira,7701ed02,7701ed02,48ab0f5f,players_ve_Venezuelan_Primera_División-06-08-2...,players_ve_venezuelan_primera_divisi_n_06_08_2...
3,20170806_7701ed02_48ab0f5f,06-08-2017,Deportivo Tachira,Atletico Venezuela,Deportivo Tachira,7701ed02,7701ed02,48ab0f5f,players_ve_Venezuelan_Primera_Division-06-08-2...,players_ve_venezuelan_primera_division_06_08_2...
4,20170806_dcbc2e42_0f8ef17f,06-08-2017,Estudiantes Merida,Dep La Guaira,Dep La Guaira,0f8ef17f,dcbc2e42,0f8ef17f,players_ve_Venezuelan_Primera_Division-06-08-2...,players_ve_venezuelan_primera_division_06_08_2...
...,...,...,...,...,...,...,...,...,...,...
195,20171008_dcbc2e42_9b0e127c,08-10-2017,Estudiantes Merida,Atletico Socopo,Estudiantes Merida,dcbc2e42,dcbc2e42,9b0e127c,players_ve_Venezuelan_Primera_Division-08-10-2...,players_ve_venezuelan_primera_division_08_10_2...
196,20171013_7701ed02_51c38375,13-10-2017,Deportivo Tachira,Dep Anzoategui,Dep Anzoategui,51c38375,7701ed02,51c38375,players_ve_Venezuelan_Primera_División-13-10-2...,players_ve_venezuelan_primera_divisi_n_13_10_2...
197,20171013_7701ed02_51c38375,13-10-2017,Deportivo Tachira,Dep Anzoategui,Dep Anzoategui,51c38375,7701ed02,51c38375,players_ve_Venezuelan_Primera_Division-13-10-2...,players_ve_venezuelan_primera_division_13_10_2...
198,20171013_7701ed02_51c38375,13-10-2017,Deportivo Tachira,Dep Anzoategui,Deportivo Tachira,7701ed02,7701ed02,51c38375,players_ve_Venezuelan_Primera_División-13-10-2...,players_ve_venezuelan_primera_divisi_n_13_10_2...


In [14]:
# Revisar source_key con guion vs letra (comparando con clave sin acentos)
import unicodedata
def normalize_source_name_ascii(name):
    if pd.isna(name):
        return name
    base = unicodedata.normalize("NFKD", str(name))
    base = "".join(ch for ch in base if not unicodedata.combining(ch))
    base = base.lower()
    base = re.sub(r"[^a-z0-9]+", "_", base)
    base = re.sub(r"_+", "_", base).strip("_")
    return base
df_key_cmp = df_totales2_dedup.dropna(subset=["match_date"]).copy()
if "source_key" not in df_key_cmp.columns:
    df_key_cmp["source_key"] = df_key_cmp["source_file"].apply(normalize_source_name)
df_key_cmp["source_key_ascii"] = df_key_cmp["source_file"].apply(normalize_source_name_ascii)
key_var = (
    df_key_cmp.groupby("source_key_ascii")["source_key"]
    .nunique()
    .sort_values(ascending=False)
)
key_var = key_var[key_var > 1]
print("Grupos donde source_key varia:", len(key_var))
if len(key_var) > 0:
    df_key_diff = df_key_cmp[df_key_cmp["source_key_ascii"].isin(key_var.index)].copy()
    df_key_diff = df_key_diff.sort_values(["source_key_ascii", "source_key", "match_date"])
    df_key_diff[["source_key_ascii", "source_key", "source_file", "match_date", "Team", "Team_id"]].head(200)

Grupos donde source_key varia: 3606


In [15]:
# Quedarse con el archivo con mas datos por source_key_ascii
df_ascii = df_totales2_dedup.dropna(subset=["match_date"]).copy()
if "source_key_ascii" not in df_ascii.columns:
    df_ascii["source_key_ascii"] = df_ascii["source_file"].apply(normalize_source_name_ascii)
cols_data_ascii = [c for c in df_ascii.columns if c not in ["source_file", "source_key_ascii"]]
df_ascii["non_nulls"] = df_ascii[cols_data_ascii].notna().sum(axis=1)
df_totales2_ascii_dedup = (
    df_ascii.sort_values(["source_key_ascii", "non_nulls"], ascending=[True, False])
    .drop_duplicates(subset=["source_key_ascii"], keep="first")
    .drop(columns=["non_nulls"])
    .reset_index(drop=True)
)
print("Filas dedup actuales:", len(df_totales2_dedup))
print("Filas dedup por source_key_ascii:", len(df_totales2_ascii_dedup))
df_totales2_ascii_dedup[["source_key_ascii", "source_file", "match_date", "Team", "Team_id"]].head(20)

Filas dedup actuales: 97152
Filas dedup por source_key_ascii: 93546


,source_key_ascii,source_file,match_date,Team,Team_id
0,players_ar_liga_profesional_argentina_01_02_20...,players_ar_Liga_Profesional_Argentina-01-02-20...,01-02-2025,Barracas Central,1d89634b
1,players_ar_liga_profesional_argentina_01_02_20...,players_ar_Liga_Profesional_Argentina-01-02-20...,01-02-2025,Banfield,06c1606c
2,players_ar_liga_profesional_argentina_01_02_20...,players_ar_Liga_Profesional_Argentina-01-02-20...,01-02-2025,Lanus,11b6dba8
3,players_ar_liga_profesional_argentina_01_02_20...,players_ar_Liga_Profesional_Argentina-01-02-20...,01-02-2025,Sarmiento–J,e9ae80b7
4,players_ar_liga_profesional_argentina_01_02_20...,players_ar_Liga_Profesional_Argentina-01-02-20...,01-02-2025,San Martin–SJ,d7a2603d
5,players_ar_liga_profesional_argentina_01_02_20...,players_ar_Liga_Profesional_Argentina-01-02-20...,01-02-2025,Rosario Central,87a920fa
6,players_ar_liga_profesional_argentina_01_03_20...,players_ar_Liga_Profesional_Argentina-01-03-20...,01-03-2025,Aldosivi,e5927bd8
7,players_ar_liga_profesional_argentina_01_03_20...,players_ar_Liga_Profesional_Argentina-01-03-20...,01-03-2025,Sarmiento–J,e9ae80b7
8,players_ar_liga_profesional_argentina_01_03_20...,players_ar_Liga_Profesional_Argentina-01-03-20...,01-03-2025,River Plate,ef99c78c
9,players_ar_liga_profesional_argentina_01_03_20...,players_ar_Liga_Profesional_Argentina-01-03-20...,01-03-2025,Estudiantes–LP,df734df9


In [16]:
# Revisar pares y anomalos en df_totales2_ascii_dedup
df_check_ascii = df_totales2_ascii_dedup.reset_index(drop=True)
mismatched_pairs_ascii = []
for i in range(0, len(df_check_ascii) - 1, 2):
    if df_check_ascii.loc[i, "match_id_new"] != df_check_ascii.loc[i + 1, "match_id_new"]:
        mismatched_pairs_ascii.append(
            (i, df_check_ascii.loc[i, "match_id_new"], df_check_ascii.loc[i + 1, "match_id_new"])
        )
print("Pares con match_id_new distinto (ascii dedup):", len(mismatched_pairs_ascii))
if mismatched_pairs_ascii:
    print("Ejemplos:")
    for row in mismatched_pairs_ascii[:10]:
        print(row)
counts_ascii = df_check_ascii["match_id_new"].value_counts(dropna=False)
anomalos_ascii = counts_ascii[(counts_ascii == 1) | (counts_ascii > 2)]
print("match_id_new anomalo (1 o >2) en ascii dedup:", len(anomalos_ascii))
anomalos_ascii.head(20)

Pares con match_id_new distinto (ascii dedup): 0
match_id_new anomalo (1 o >2) en ascii dedup: 75


,count
match_id_new,
20210403_cff3d9bb_60c6b05f,4
20201004_33c895d4_60c6b05f,4
20201215_b8fd03ef_60c6b05f,4
20210130_60c6b05f_fd962109,4
20180303_2abfe087_60c6b05f,4
20180421_60c6b05f_822bd0ba,4
20210425_8602292d_60c6b05f,4
20171104_f5922ca5_60c6b05f,4
20201019_60c6b05f_943e8050,4


In [17]:
# Mostrar filas anómalas en df_totales2_ascii_dedup
anom_ids_ascii = anomalos_ascii.index.tolist()
cols_show_ascii = [
    "match_id_new",
    "match_date",
    "home_team",
    "away_team",
    "Team",
    "Team_id",
    "team_id_home",
    "team_id_away",
    "source_file",
    "source_key_ascii",
]
df_anom_ascii = (
    df_totales2_ascii_dedup[df_totales2_ascii_dedup["match_id_new"].isin(anom_ids_ascii)]
    .sort_values(["match_id_new", "Team"])
.reset_index(drop=True)
)
df_anom_ascii[cols_show_ascii].head(200)

,match_id_new,match_date,home_team,away_team,Team,Team_id,team_id_home,team_id_away,source_file,source_key_ascii
0,20170812_60c6b05f_4ba7cbea,12-08-2017,West Brom,Bournemouth,Bournemouth,4ba7cbea,60c6b05f,4ba7cbea,players_eng_Premier_League-12-08-2017-West_Bro...,players_eng_premier_league_12_08_2017_west_bro...
1,20170812_60c6b05f_4ba7cbea,12-08-2017,West Bromwich Albion,Bournemouth,Bournemouth,4ba7cbea,60c6b05f,4ba7cbea,players_eng_Premier_League-12-08-2017-West_Bro...,players_eng_premier_league_12_08_2017_west_bro...
2,20170812_60c6b05f_4ba7cbea,12-08-2017,West Brom,Bournemouth,West Brom,60c6b05f,60c6b05f,4ba7cbea,players_eng_Premier_League-12-08-2017-West_Bro...,players_eng_premier_league_12_08_2017_west_bro...
3,20170812_60c6b05f_4ba7cbea,12-08-2017,West Bromwich Albion,Bournemouth,West Bromwich Albion,60c6b05f,60c6b05f,4ba7cbea,players_eng_Premier_League-12-08-2017-West_Bro...,players_eng_premier_league_12_08_2017_west_bro...
4,20170819_943e8050_60c6b05f,19-08-2017,Burnley,West Brom,Burnley,943e8050,943e8050,60c6b05f,players_eng_Premier_League-19-08-2017-Burnley-...,players_eng_premier_league_19_08_2017_burnley_...
...,...,...,...,...,...,...,...,...,...,...
195,20201206_60c6b05f_47c64c55,06-12-2020,West Bromwich Albion,Crystal Palace,West Bromwich Albion,60c6b05f,60c6b05f,47c64c55,players_eng_Premier_League-06-12-2020-West_Bro...,players_eng_premier_league_06_12_2020_west_bro...
196,20201212_b2b47a98_60c6b05f,12-12-2020,Newcastle United,West Brom,Newcastle United,b2b47a98,b2b47a98,60c6b05f,players_eng_Premier_League-12-12-2020-Newcastl...,players_eng_premier_league_12_12_2020_newcastl...
197,20201212_b2b47a98_60c6b05f,12-12-2020,Newcastle United,West Bromwich Albion,Newcastle United,b2b47a98,b2b47a98,60c6b05f,players_eng_Premier_League-12-12-2020-Newcastl...,players_eng_premier_league_12_12_2020_newcastl...
198,20201212_b2b47a98_60c6b05f,12-12-2020,Newcastle United,West Brom,West Brom,60c6b05f,b2b47a98,60c6b05f,players_eng_Premier_League-12-12-2020-Newcastl...,players_eng_premier_league_12_12_2020_newcastl...


In [18]:
# Contar West Brom vs West Bromwich y duplicados por match_id_new
df_wb = df_totales2_ascii_dedup.copy()
name_cols = ["home_team", "away_team", "Team"]
def _count_name(df, name):
    return df[name_cols].apply(lambda s: s.str.contains(name, case=False, na=False)).any(axis=1).sum()
print("Conteo West Brom:", _count_name(df_wb, "West Brom"))
print("Conteo West Bromwich:", _count_name(df_wb, "West Bromwich"))
wb_mask = df_wb[name_cols].apply(lambda s: s.str.contains("West Brom", case=False, na=False)).any(axis=1)
wbw_mask = df_wb[name_cols].apply(lambda s: s.str.contains("West Bromwich", case=False, na=False)).any(axis=1)
df_wb_dups = df_wb[wb_mask | wbw_mask].copy()
counts_wb = df_wb_dups["match_id_new"].value_counts()
dup_match_ids_wb = counts_wb[counts_wb > 2]
print("match_id_new duplicados (>2) con West Brom/Bromwich:", len(dup_match_ids_wb))
dup_match_ids_wb.head(20)

Conteo West Brom: 302
Conteo West Bromwich: 150
match_id_new duplicados (>2) con West Brom/Bromwich: 75


,count
match_id_new,
20180102_7c21e445_60c6b05f,4
20210102_60c6b05f_18bb7c10,4
20210202_1df6b87e_60c6b05f,4
20201102_fd962109_60c6b05f,4
20171202_60c6b05f_47c64c55,4
20180203_60c6b05f_33c895d4,4
20180303_2abfe087_60c6b05f,4
20210403_cff3d9bb_60c6b05f,4
20210503_60c6b05f_8cec06e1,4


In [19]:
# Normalizar West Bromwich/West Brom Albion -> West Brom en Team/home_team/away_team
df_wb_norm = df_totales2_ascii_dedup.copy()
cols_team = ["Team", "home_team", "away_team"]
for col in cols_team:
    df_wb_norm[col] = df_wb_norm[col].str.replace(
        r"\bWest Bromwich\b", "West Brom", case=False, regex=True
    )
    df_wb_norm[col] = df_wb_norm[col].str.replace(
        r"\bWest Brom Albion\b", "West Brom", case=False, regex=True
    )
df_wb_norm[cols_team].head()

,Team,home_team,away_team
0,Barracas Central,Barracas Central,Banfield
1,Banfield,Barracas Central,Banfield
2,Lanus,Lanus,Sarmiento–J
3,Sarmiento–J,Lanus,Sarmiento–J
4,San Martin–SJ,San Martin–SJ,Rosario Central


In [20]:
# Deduplicar por (Team_id, match_id_new) manteniendo la fila con mas datos
def dedup_by_team_match(df, team_col="Team_id", match_col="match_id_new"):
    df = df.copy()
    cols_data_match = [c for c in df.columns if c not in [team_col, match_col]]
    df["non_nulls"] = df[cols_data_match].notna().sum(axis=1)
    df = df.sort_values([match_col, team_col, "non_nulls"], ascending=[True, True, False])
    df_dedup = (
        df.drop_duplicates(subset=[match_col, team_col], keep="first")
        .drop(columns=["non_nulls"])
        .reset_index(drop=True)
    )
    return df_dedup
df_wb_match_dedup = dedup_by_team_match(df_wb_norm, team_col="Team_id", match_col="match_id_new")
print("Filas antes:", len(df_wb_norm))
print("Filas despues (Team_id + match_id_new):", len(df_wb_match_dedup))

Filas antes: 93546
Filas despues (Team_id + match_id_new): 93396


In [21]:
# Revisar anomalias de match_id_new despues del dedup Team_id + match_id_new
df_check_wb = df_wb_match_dedup.reset_index(drop=True)
counts_wb_new = df_check_wb["match_id_new"].value_counts(dropna=False)
anomalos_wb = counts_wb_new[(counts_wb_new == 1) | (counts_wb_new > 2)]
print("match_id_new anomalo (1 o >2) en wb_match_dedup:", len(anomalos_wb))
anomalos_wb.head(20)

match_id_new anomalo (1 o >2) en wb_match_dedup: 0


,count
match_id_new,


In [22]:
# Verificar que cada match_id_new tenga exactamente 2 filas (local/visita)
counts_match = df_wb_match_dedup["match_id_new"].value_counts(dropna=False)
bad_match = counts_match[counts_match != 2]
print("match_id_new con conteo distinto de 2:", len(bad_match))
bad_match.head(20)

match_id_new con conteo distinto de 2: 0


,count
match_id_new,


In [23]:
# Contar match_id_new unicos en df_totales2 y df_wb_match_dedup
count_totales2 = df_totales2["match_id_new"].nunique(dropna=False)
count_wb_match = df_wb_match_dedup["match_id_new"].nunique(dropna=False)
print("Unicos match_id_new en df_totales2:", count_totales2)
print("Unicos match_id_new en df_wb_match_dedup:", count_wb_match)

Unicos match_id_new en df_totales2: 46698
Unicos match_id_new en df_wb_match_dedup: 46698


In [24]:
# Encontrar match_id_new que estan en df_totales2 pero no en df_wb_match_dedup
set_totales2 = set(df_totales2["match_id_new"].dropna())
set_wb_match = set(df_wb_match_dedup["match_id_new"].dropna())
missing_match_ids = sorted(set_totales2 - set_wb_match)
print("match_id_new perdidos:", len(missing_match_ids))
missing_match_ids[:20]

match_id_new perdidos: 0


[]

In [25]:
# Columnas del dataframe final sin duplicados
df_wb_match_dedup.columns.tolist()

['Team_id',
 'Goals',
 'Assists',
 'GoalsPen',
 'AttPen',
 'Shots',
 'ShTarget',
 'YellowCard',
 'RedCard',
 'Fouls',
 'FoulsD',
 'Offside',
 'Crosses',
 'TacklesW',
 'Interceptions',
 'OwnGoals',
 'PenKickGen',
 'PenKickCon',
 'home_team',
 'away_team',
 'Team',
 'match_id',
 'source_file',
 'match_id_new',
 'match_date',
 'team_id_home',
 'team_id_away',
 'source_key',
 'source_key_ascii']

In [26]:
# Contar West Brom vs West Bromwich y duplicados por match_id_new
df_wb2 = df_wb_norm.copy()
name_cols = ["home_team", "away_team", "Team"]
def _count_name(df, name):
    return df[name_cols].apply(lambda s: s.str.contains(name, case=False, na=False)).any(axis=1).sum()
print("Conteo West Brom:", _count_name(df_wb2, "West Brom"))
print("Conteo West Bromwich:", _count_name(df_wb2, "West Bromwich"))
wb_mask = df_wb2[name_cols].apply(lambda s: s.str.contains("West Brom", case=False, na=False)).any(axis=1)
wbw_mask = df_wb2[name_cols].apply(lambda s: s.str.contains("West Bromwich", case=False, na=False)).any(axis=1)
df_wb_dups = df_wb2[wb_mask | wbw_mask].copy()
counts_wb = df_wb_dups["match_id_new"].value_counts()
dup_match_ids_wb = counts_wb[counts_wb > 2]
print("match_id_new duplicados (>2) con West Brom/Bromwich:", len(dup_match_ids_wb))
dup_match_ids_wb.head(20)

Conteo West Brom: 302
Conteo West Bromwich: 0
match_id_new duplicados (>2) con West Brom/Bromwich: 75


,count
match_id_new,
20180102_7c21e445_60c6b05f,4
20210102_60c6b05f_18bb7c10,4
20210202_1df6b87e_60c6b05f,4
20201102_fd962109_60c6b05f,4
20171202_60c6b05f_47c64c55,4
20180203_60c6b05f_33c895d4,4
20180303_2abfe087_60c6b05f,4
20210403_cff3d9bb_60c6b05f,4
20210503_60c6b05f_8cec06e1,4


In [27]:
# Conteos exactos de West Brom y West Bromwich (sin substring)
df_wb3 = df_wb_norm.copy()
cols_team = ["home_team", "away_team", "Team"]
def _count_exact(df, name):
    return df[cols_team].apply(lambda s: s.str.lower().eq(name.lower())).any(axis=1).sum()
def _count_word(df, name):
    return df[cols_team].apply(
        lambda s: s.str.contains(fr"\b{name}\b", case=False, na=False, regex=True)
    ).any(axis=1).sum()
print("Exacto West Brom:", _count_exact(df_wb3, "West Brom"))
print("Exacto West Bromwich:", _count_exact(df_wb3, "West Bromwich"))
print("Word-boundary West Brom:", _count_word(df_wb3, "West Brom"))
print("Word-boundary West Bromwich:", _count_word(df_wb3, "West Bromwich"))

Exacto West Brom: 302
Exacto West Bromwich: 0
Word-boundary West Brom: 302
Word-boundary West Bromwich: 0


In [28]:
# Nombres unicos que contienen "West Brom" (en Team/home_team/away_team)
df_names = df_wb_norm.copy()
cols_team = ["Team", "home_team", "away_team"]
mask_wb = df_names[cols_team].apply(
    lambda s: s.str.contains("West Brom", case=False, na=False)
).any(axis=1)
unique_names = (
    pd.unique(df_names.loc[mask_wb, cols_team].values.ravel())
)
unique_names = [n for n in unique_names if isinstance(n, str) and "west brom" in n.lower()]
unique_names = sorted(set(unique_names))
print("Cantidad de nombres unicos:", len(unique_names))
unique_names[:50]

Cantidad de nombres unicos: 1


['West Brom']

In [29]:
# Crear columna competition desde source_file/source_key_ascii
def extract_competition(source_file):
    if pd.isna(source_file):
        return None
    name = os.path.basename(str(source_file))
    base = re.sub(r"_summary\.csv$", "", name)
    m = re.match(r"^(players_.+?)-\d{2}-\d{2}-\d{4}", base)
    if not m:
        return None
    competition = m.group(1)
    competition = re.sub(r"^players_", "", competition)
    # Quitar prefijo pais_ para ligas nacionales (pp_ o eng_)
    competition = re.sub(r"^(eng|[a-z]{2})_", "", competition)
    return competition
df_wb_match_dedup2 = df_wb_match_dedup.copy()
df_wb_match_dedup2["competition"] = df_wb_match_dedup2["source_file"].apply(extract_competition)
df_wb_match_dedup2["competition"].value_counts().head(30)

,count
competition,
Major_League_Soccer,7696
Premier_League,6454
La_Liga,6416
Ligue_1,6066
Liga_MX,5720
J1_League,5598
Bundesliga,5162
Serie_A,4280
Saudi_Pro_League,4186


In [30]:
# Filtrar competencias que no se usan
drop_competitions = ["Trophee_des_Champions", "Premiere_Ligue"]
df_wb_match_dedup3 = df_wb_match_dedup2[~df_wb_match_dedup2["competition"].isin(drop_competitions)].copy()
print("Filas antes:", len(df_wb_match_dedup2))
print("Filas despues:", len(df_wb_match_dedup3))
df_wb_match_dedup3["competition"].value_counts().head(30)

Filas antes: 93396
Filas despues: 93002


,count
competition,
Major_League_Soccer,7696
Premier_League,6454
La_Liga,6416
Ligue_1,6066
Liga_MX,5720
J1_League,5598
Bundesliga,5162
Serie_A,4280
Saudi_Pro_League,4186


In [31]:
# Equipos y Team_id para Serie A y Serie A (Série A) antes de filtrar
import unicodedata
def _normalize_comp(value):
    if pd.isna(value):
        return None
    base = unicodedata.normalize("NFKD", str(value))
    base = "".join(ch for ch in base if not unicodedata.combining(ch))
    return base.lower()
def extract_competition_raw(source_file):
    if pd.isna(source_file):
        return None
    name = os.path.basename(str(source_file))
    base = re.sub(r"_summary\.csv$", "", name)
    m = re.match(r"^(players_.+?)-\d{2}-\d{2}-\d{4}", base)
    if not m:
        return None
    competition = m.group(1)
    competition = re.sub(r"^players_", "", competition)
    return competition
df_comp_check = df_wb_match_dedup3.copy()
df_comp_check["competition_raw"] = df_comp_check["source_file"].apply(extract_competition_raw)
df_comp_check["competition_raw_norm"] = df_comp_check["competition_raw"].apply(_normalize_comp)
df_serie_a = df_comp_check[df_comp_check["competition_raw_norm"].str.endswith("serie_a", na=False)].copy()
print("Competencias raw encontradas:", df_serie_a["competition_raw"].dropna().unique())
teams_serie_a = (
    df_serie_a[["competition_raw", "Team", "Team_id"]]
    .dropna(subset=["Team", "Team_id"])
    .drop_duplicates()
    .sort_values(["competition_raw", "Team"])
    .reset_index(drop=True)
 )
teams_serie_a

Competencias raw encontradas: ['br_Série_A' 'it_Serie_A']


,competition_raw,Team,Team_id
0,br_Série_A,America–MG,1f68d780
1,br_Série_A,Athletico Paranaense,2091c619
2,br_Série_A,Atletico Goianiense,32d508ca
3,br_Série_A,Atletico Mineiro,422bb734
4,br_Série_A,Avai,f205258a
...,...,...,...
70,it_Serie_A,Spezia,68449f6d
71,it_Serie_A,Ternana,eb8e3e81
72,it_Serie_A,Torino,105360fe
73,it_Serie_A,Udinese,04eea015


In [32]:
# Lista completa Team/Team_id para it_Serie_A
teams_it_serie_a = (
    df_comp_check[df_comp_check["competition_raw"] == "it_Serie_A"][["Team", "Team_id"]]
    .dropna(subset=["Team", "Team_id"])
    .drop_duplicates()
    .sort_values("Team")
    .reset_index(drop=True)
 )
teams_it_serie_a

,Team,Team_id
0,AS Roma,02f8d026
1,ASD Calcio Pomigliano,9f18ced7
2,Atalanta,922493f3
3,Bologna,1d8099f8
4,Cagliari,c4260e09
5,Como,28c9c3cd
6,Cremonese,9aad3a77
7,Empoli,a3d88bd8
8,Empoli,b72bc283
9,FC Como Women,756c9acb


In [33]:
# Paso 1: Definir los nombres de los equipos femeninos italianos
women_teams = ["ASD Calcio Pomigliano", "FC Como Women", "Ternana", "AS Roma", "AGSM Verona", "FC Como Women", "Orobica Calcio", "Pink Bari", "San Marino", "Tavagnacco"]

# Paso 2: Obtener los Team_id de esos equipos en teams_it_serie_a
women_ids = teams_it_serie_a[teams_it_serie_a["Team"].isin(women_teams)]["Team_id"].dropna().unique().tolist()
print("Team_id femeninos italianos:", women_ids)

# ===== MODIFICACIÓN: AÑADIR LISTA MANUAL DEPURADA =====
# Lista depurada (SOLO FEMENINOS) según tus anotaciones
ids_drop_match = [
    '02f8d026',   # Roma fem
    '06d05f19',   # Internazionale fem
    '20f5bd86',   # San Marino fem
    '28d9b675',   # Paris Fem
    '331a33a7',   # Tavagnacco fem
    '39abd503',   # Glasgowfem
    '3d377a8c',   # St Polten fem
    '54da9a57',   # Milan Fem
    '658bf2de',   # Genoa Fem
    '19db4c32',   # Lazio fem
    '756c9acb',   # Como Women
    '80595417',   # PSGFem
    '9f18ced7',   # Pomigliano
    'adba8c92',   # Orobica
    'be35aa4d',   # Pink Bari
    'c1cf4fdc',   # Sparta Fem
    'cc6ecd46',   # Chievo fem
    'd42d8b2f',   # Fiorentina fem
    '828ce826',   # Napoli Fem
    'd59e6510',   # AGSMVerona
    'e83b563b',   # Servette fem
    'eb8e3e81',   # Ternana
    'b72bc283',   # Empoli Fem
    '613577b8',   # Juventus Fem 
    'a8e83f23',   # Sassuolo Fem 
    'f7d05cb5',   # Parma Fem 
    'd35ed67b',   # Hellas Verona fem
    '58e367c3',   # Sampdoria
    'a8e83f23',   # Sassuolo
]
# Unir women_ids + lista manual en un solo set (IDs únicos femeninos)
all_team_ids = sorted(set(str(x) for x in women_ids) | set(str(x) for x in ids_drop_match))
print(f"Total IDs femeninos únicos: {len(all_team_ids)}")

# Paso 3: Buscar en df_comp_check todas las filas donde Team_id sea uno de esos ids
# mask_women = df_comp_check["Team_id"].isin(women_ids)
#df_women_matches = df_comp_check[mask_women].copy()

# Paso 4: Recolectar todos los home_team_id y away_team_id junto con home_team y away_team
#home_ids = set(df_women_matches["team_id_home"].dropna().unique())
#home_names = set(df_women_matches["home_team"].dropna().unique())
#away_ids = set(df_women_matches["team_id_away"].dropna().unique())
#away_names = set(df_women_matches["away_team"].dropna().unique())

## Unir todos los ids y nombres en listas sin repetición
#all_team_ids = sorted(str(x) for x in home_ids.union(away_ids))
#all_team_names = sorted(home_names.union(away_names))

#print("Todos los Team_id detectados:", all_team_ids)
#print("Todos los nombres detectados:", all_team_names)

# Mostrar correspondencia entre Team_id y Team SOLO para equipos de Serie A
serie_a_mask = (df_comp_check['competition_raw'] == 'it_Serie_A')
df_team_lookup = df_comp_check[serie_a_mask][['Team_id', 'Team']].dropna().drop_duplicates()
df_team_lookup = df_team_lookup[df_team_lookup['Team_id'].isin(all_team_ids)]
df_team_lookup = df_team_lookup.sort_values('Team').reset_index(drop=True)

print('Correspondencia Team_id <-> Team (solo Serie A):')
display(df_team_lookup)

Team_id femeninos italianos: ['02f8d026', '9f18ced7', '756c9acb', 'eb8e3e81']
Total IDs femeninos únicos: 28
Correspondencia Team_id <-> Team (solo Serie A):


,Team_id,Team
0,02f8d026,AS Roma
1,9f18ced7,ASD Calcio Pomigliano
2,b72bc283,Empoli
3,756c9acb,FC Como Women
4,658bf2de,Genoa
5,d35ed67b,Hellas Verona
6,06d05f19,Internazionale
7,613577b8,Juventus
8,19db4c32,Lazio
9,54da9a57,Milan


In [34]:
# IDs de equipos femeninos a investigar
ids_ejemplo = ['756c9acb', '9f18ced7']  # Como Women y Pomigliano

for fid in ids_ejemplo:
    print(f"\n{'='*60}")
    print(f"ANÁLISIS PARA ID: {fid}")
    print('='*60)
    
    # 1. Filas donde el equipo femenino aparece como Team_id (archivo de ese equipo)
    mask_team = df_comp_check['Team_id'].astype(str) == fid
    print(f"Filas con Team_id = {fid}: {mask_team.sum()}")
    
    # 2. Filas donde aparece como local o visitante
    mask_home = df_comp_check['team_id_home'].astype(str) == fid
    mask_away = df_comp_check['team_id_away'].astype(str) == fid
    print(f"Filas donde {fid} es local: {mask_home.sum()}, visitante: {mask_away.sum()}")
    
    # 3. Partidos únicos donde aparece
    match_ids = set(df_comp_check.loc[mask_team | mask_home | mask_away, 'match_id_new'].dropna())
    print(f"Número de partidos únicos con {fid}: {len(match_ids)}")
    
    # 4. Obtener todas las filas de esos partidos
    partidos = df_comp_check[df_comp_check['match_id_new'].isin(match_ids)]
    
    # 5. IDs y nombres de todos los equipos que participan en esos partidos
    all_home_ids = set(partidos['team_id_home'].dropna().astype(str).unique())
    all_away_ids = set(partidos['team_id_away'].dropna().astype(str).unique())
    all_team_ids = all_home_ids.union(all_away_ids)
    all_team_names = set(partidos['home_team'].dropna().unique()).union(set(partidos['away_team'].dropna().unique()))
    
    print(f"\nIDs de equipos que participan en partidos con {fid}:")
    print(sorted(all_team_ids))
    print(f"\nNombres de equipos que participan:")
    print(sorted(all_team_names))
    
    # 6. Ver cuántos de esos IDs están en nuestra lista de femeninos
    #femeninos_en_partidos = all_team_ids.intersection(all_female_ids)
    #print(f"\nDe estos, los que están en all_female_ids: {femeninos_en_partidos}")
    
    # 7. Ver si hay IDs que deberían ser masculinos pero están en all_female_ids
    #posibles_masculinos = [mid for mid in all_team_ids if mid not in femeninos_en_partidos]
    #print(f"IDs que NO están en all_female_ids (posibles masculinos): {posibles_masculinos}")
    
    # 8. Mostrar una muestra de los partidos
    #print("\nMuestra de partidos (hasta 5):")
    #display(partidos[['match_id_new', 'home_team', 'away_team', 'team_id_home', 'team_id_away', 'Team_id']].head(5))


ANÁLISIS PARA ID: 756c9acb
Filas con Team_id = 756c9acb: 87
Filas donde 756c9acb es local: 88, visitante: 86
Número de partidos únicos con 756c9acb: 87

IDs de equipos que participan en partidos con 756c9acb:
['02f8d026', '06d05f19', '19db4c32', '2998cfbf', '54da9a57', '58e367c3', '613577b8', '756c9acb', '828ce826', '9f18ced7', 'a8e83f23', 'c99ff6e5', 'eb8e3e81', 'f7d05cb5']

Nombres de equipos que participan:
['AS Roma', 'ASD Calcio Pomigliano', 'FC Como Women', 'Fiorentina', 'Genoa', 'Internazionale', 'Juventus', 'Lazio', 'Milan', 'Napoli', 'Parma', 'Sampdoria', 'Sassuolo', 'Ternana']

ANÁLISIS PARA ID: 9f18ced7
Filas con Team_id = 9f18ced7: 68
Filas donde 9f18ced7 es local: 68, visitante: 68
Número de partidos únicos con 9f18ced7: 68

IDs de equipos que participan en partidos con 9f18ced7:
['02f8d026', '06d05f19', '19db4c32', '54da9a57', '58e367c3', '613577b8', '756c9acb', '828ce826', '9f18ced7', 'a8e83f23', 'b72bc283', 'c99ff6e5', 'd35ed67b', 'f7d05cb5']

Nombres de equipos que pa

In [35]:
# ============================================================
# FILTRADO COMPLETO DE EQUIPOS FEMENINOS
# (Solo por team_id_home y team_id_away, NO por Team_id)
# ============================================================

# 1. Unificar todos los IDs femeninos (desde women_ids y lista manual)
all_female_ids = set(str(x) for x in women_ids) | set(str(x) for x in ids_drop_match)
print(f"Total IDs femeninos a eliminar: {len(all_female_ids)}")
print(f"IDs femeninos a eliminar: {all_female_ids}")

# 2. Máscara: eliminar si team_id_home o team_id_away están en la lista
# (NO filtramos por Team_id, porque Team_id puede tener IDs femeninos asignados a equipos masculinos)
mask_ids = (
    df_comp_check['team_id_home'].astype(str).isin(all_female_ids) |
    df_comp_check['team_id_away'].astype(str).isin(all_female_ids)
)

df_comp_check_limpio = df_comp_check[~mask_ids].copy()
print(f"Filas originales: {len(df_comp_check):,}")
print(f"Eliminadas por IDs (home/away): {mask_ids.sum():,}")
print(f"Restantes (antes de match_id_new): {len(df_comp_check_limpio):,}")

# 3. Drop adicional por match_id_new (por si algún partido completo se cuela)
removed_extra = 0
if "match_id_new" in df_comp_check_limpio.columns:
    pattern = "|".join(ids_drop_match)
    mask_match = df_comp_check_limpio["match_id_new"].astype(str).str.contains(pattern, na=False)
    removed_extra = int(mask_match.sum())
    df_comp_check_limpio = df_comp_check_limpio[~mask_match].copy()

print(f"Eliminados adicionales por match_id_new: {removed_extra:,}")
print(f"Filas finales: {len(df_comp_check_limpio):,}")

Total IDs femeninos a eliminar: 28
IDs femeninos a eliminar: {'20f5bd86', '19db4c32', 'f7d05cb5', '39abd503', 'adba8c92', 'eb8e3e81', 'e83b563b', '331a33a7', '756c9acb', 'b72bc283', '02f8d026', '80595417', '54da9a57', '58e367c3', '28d9b675', 'c1cf4fdc', 'd42d8b2f', '9f18ced7', '3d377a8c', 'd59e6510', 'a8e83f23', '828ce826', 'be35aa4d', 'd35ed67b', '06d05f19', 'cc6ecd46', '613577b8', '658bf2de'}


Filas originales: 93,002
Eliminadas por IDs (home/away): 1,712
Restantes (antes de match_id_new): 91,290
Eliminados adicionales por match_id_new: 0
Filas finales: 91,290


In [36]:
# Verificar que los IDs masculinos que te interesan no estén en la lista de eliminados
masculinos_interes = ['a3d88bd8', 'd48ad4ff', 'dc56fe14', 'e0652b02', 'e2befd26', '7213da33']
for mid in masculinos_interes:
    if mid in all_female_ids:
        print(f"¡Cuidado! {mid} está en la lista de femeninos")
    else:
        print(f"{mid} no está en la lista, bien.")

a3d88bd8 no está en la lista, bien.
d48ad4ff no está en la lista, bien.
dc56fe14 no está en la lista, bien.
e0652b02 no está en la lista, bien.
e2befd26 no está en la lista, bien.
7213da33 no está en la lista, bien.


In [37]:
# Aseguramos que all_team_ids_str sea un set para búsqueda ultra rápida
all_team_ids_set = set(str(x) for x in all_team_ids)

# Creamos la máscara comparando directamente las columnas de ID
# Esto es miles de veces más rápido que un .apply() con bucle for
mask_excluir = (
    df_comp_check['Team_id'].astype(str).isin(all_team_ids_set) |
    df_comp_check['team_id_home'].astype(str).isin(all_team_ids_set) | 
    df_comp_check['team_id_away'].astype(str).isin(all_team_ids_set)
)

df_comp_check_limpio = df_comp_check[~mask_excluir].copy()

print(f"Filas originales: {len(df_comp_check)}")
print(f"Filas eliminadas: {mask_excluir.sum()}")
print(f"Filas restantes: {len(df_comp_check_limpio)}")

Filas originales: 93002
Filas eliminadas: 1232
Filas restantes: 91770


In [38]:
# ===== SOLO DROP POR match_id_new (ya que los IDs ya fueron filtrados en celda 3) =====
# Lista depurada (misma que en celda 2)
ids_drop_match = [
    '02f8d026', '06d05f19', '20f5bd86', '28d9b675', '331a33a7', '39abd503',
    '3d377a8c', '54da9a57', '658bf2de', '7213da33', '756c9acb', '80595417',
    '9f18ced7', 'adba8c92', 'be35aa4d', 'c1cf4fdc', 'cc6ecd46', 'd42d8b2f',
    'd48ad4ff', 'd59e6510', 'e83b563b', 'eb8e3e81', 'b72bc283', '613577b8',
    'a8e83f23', 'f7d05cb5'
]

antes = len(df_comp_check_limpio)
removed_extra = 0
if "match_id_new" in df_comp_check_limpio.columns:
    pattern = "|".join(ids_drop_match)
    mask_match = df_comp_check_limpio["match_id_new"].astype(str).str.contains(pattern, na=False)
    removed_extra = int(mask_match.sum())
    df_comp_check_limpio = df_comp_check_limpio[~mask_match].copy()

print(f"--- Auditoría de Limpieza (drop por match_id_new) ---")
print(f"Filas antes del drop adicional: {antes}")
print(f"Eliminadas por match_id_new: {removed_extra}")
print(f"Filas finales: {len(df_comp_check_limpio)}")

--- Auditoría de Limpieza (drop por match_id_new) ---
Filas antes del drop adicional: 91770
Eliminadas por match_id_new: 1374
Filas finales: 90396


In [39]:
# 1. Normalización de tipos (Zero-overhead lookup)
# Usamos un set de strings para búsqueda O(1)
#ids_para_eliminar = set(map(str, all_team_ids))

# 2. Máscara Booleana Vectorizada
# Evitamos .apply() porque es un loop encubierto que Python odia.
# Usamos .isin() que está optimizado en C.
#mask_home = df_comp_check['team_id_home'].astype(str).isin(ids_para_eliminar)
#mask_away = df_comp_check['team_id_away'].astype(str).isin(ids_para_eliminar)

# 3. Operación lógica de conjunto
#mask_final = mask_home | mask_away

# 4. Generación del DataFrame Limpio
#df_comp_check_limpio = df_comp_check[~mask_final].copy()

# 5. Drop adicional por match_id_new (date_hometeamid_awayteamid)
#ids_drop_match = [
    #'02f8d026', '054efa67', '06d05f19', '0e72edf2', '206d90db', '20f5bd86', '28d9b675',
    #'331a33a7', '39abd503', '3d377a8c', '421387cf', '4573a712', '4eaa11d7', '54da9a57',
    #'658bf2de', '7213da33', '756c9acb', '80595417', '8b4cbfb9', '8ff9e3b3', '922493f3',
    #'99f258c2', '9f18ced7', 'a3d88bd8', 'adba8c92', 'be35aa4d', 'c1cf4fdc', 'cc6ecd46',
    #'d42d8b2f', 'd48ad4ff', 'd53c0b06', 'd59e6510', 'dc56fe14', 'e0652b02', 'e2befd26',
    #'e83b563b', 'eab4234c', 'eb8e3e81', 'ecd11ca2'
#]

#removed_extra = 0
#if "match_id_new" in df_comp_check_limpio.columns:
    #pattern = "|".join(ids_drop_match)
    #mask_match = df_comp_check_limpio["match_id_new"].astype(str).str.contains(pattern, na=False)
    #removed_extra = int(mask_match.sum())
    #df_comp_check_limpio = df_comp_check_limpio[~mask_match].copy()

# Reporte de integridad
#print(f"--- Auditoría de Limpieza ---")
#print(f"Registros procesados: {len(df_comp_check):,}")
#print(f"Registros eliminados: {(mask_final.sum() + removed_extra):,}")
#print(f"Registros restantes:  {len(df_comp_check_limpio):,}")

In [40]:
# 1. Definir el patrón de búsqueda (Regex) para mayor flexibilidad
# Buscamos 'Champions League' ignorando mayúsculas/minúsculas
target_comp = "Champions_League"

# 2. Filtrado y Agregación Eficiente
# Filtramos, seleccionamos columnas y eliminamos duplicados en un solo paso de memoria
df_champions_lookup = (
    df_comp_check[df_comp_check["competition_raw"].str.contains(target_comp, case=False, na=False)]
    [["Team_id", "Team", "competition_raw"]]
    .drop_duplicates()
    .sort_values(by=["competition_raw", "Team"])
)

# 3. Formateo de salida para inspección rápida
if not df_champions_lookup.empty:
    print(f"--- Equipos Detectados en {target_comp} ---")
    print(f"Total registros únicos: {len(df_champions_lookup)}")
    
    # Usamos display para ver la tabla formateada si estás en Jupyter
    display(df_champions_lookup)
else:
    print(f"Alerta: No se encontraron registros con '{target_comp}' en competition_raw.")

# 4. Proactividad Codex: Validación de segmentación
# Vamos a ver si existen variantes que incluyan 'Women' o 'Fem' para ayudarte a identificar rápido
women_variants = df_champions_lookup[
    df_champions_lookup["competition_raw"].str.contains("Women|Fem|Wom", case=False, na=False)
]

if not women_variants.empty:
    print("\n--- Posibles equipos femeninos identificados por nombre de competición ---")
    display(women_variants)

--- Equipos Detectados en Champions_League ---
Total registros únicos: 372


,Team_id,Team,competition_raw
45705,922c3cf8,AS Cavaly,CONCACAF_Champions_League
35523,b93b7301,Alajuelense,CONCACAF_Champions_League
25330,74480d38,Alianza FC,CONCACAF_Champions_League
25342,18d3c3a3,America,CONCACAF_Champions_League
35504,1cde5599,Arcahaie FC,CONCACAF_Champions_League
...,...,...,...
50588,2a6cfcb2,Zrinjski Mostar,Champions_League
50774,22a5a99c,Zurich,Champions_League
10732,82a7cabd,Zvezda Perm,Champions_League
10702,6d12ffd1,Þor KA,Champions_League


In [41]:
# 1. Definición de filtros de exclusión (escalable)
competiciones_a_excluir = ['CONCACAF_Champions_League']

# 2. Filtrado vectorizado con lógica inversa (~ e .isin)
# Esto garantiza que solo nos quedemos con lo que NO está en la lista de exclusión
df_champions_filtrado = df_champions_lookup[
    ~df_champions_lookup["competition_raw"].isin(competiciones_a_excluir)
].copy()

# 3. Reporte de limpieza
print(f"--- Limpieza de Competiciones Realizada ---")
print(f"Registros antes del filtro: {len(df_champions_lookup)}")
print(f"Registros después del filtro: {len(df_champions_filtrado)}")
print(f"Competiciones restantes: {df_champions_filtrado['competition_raw'].unique().tolist()}")

# Mostrar resultados
display(df_champions_filtrado)

--- Limpieza de Competiciones Realizada ---
Registros antes del filtro: 372
Registros después del filtro: 329
Competiciones restantes: ['Champions_League']


,Team_id,Team,competition_raw
39127,ef289882,1 FC Slovacko,Champions_League
28332,15e031af,AC Omonia,Champions_League
74989,d6a485e1,AC Virtus,Champions_League
9587,d5348c80,AEK Athens,Champions_League
50768,01689b1a,AEK Larnaca FC,Champions_League
...,...,...,...
50588,2a6cfcb2,Zrinjski Mostar,Champions_League
50774,22a5a99c,Zurich,Champions_League
10732,82a7cabd,Zvezda Perm,Champions_League
10702,6d12ffd1,Þor KA,Champions_League


In [42]:
# 1. Agregación de métricas de volumen por equipo en Champions_League
# Contamos cuántas veces aparece cada Team_id para ver su "peso" en el dataset
df_audit = (
    df_comp_check[df_comp_check["competition_raw"] == "Champions_League"]
    .groupby(["Team_id", "Team"])
    .size()
    .reset_index(name="match_count")
)

# 2. Flag Proactivo: Detección por palabras clave en el nombre del Club
# Muchos equipos femeninos incluyen "Women", "WFC", "Ladies" o "Feminnile"
keywords = r"Women|WFC|Ladies|Fem|Wom|Dames|Frauen"
df_audit["probable_femenino"] = df_audit["Team"].str.contains(keywords, case=False, na=False)

# 3. Ordenar por probabilidad y nombre
df_audit = df_audit.sort_values(by=["probable_femenino", "Team"], ascending=[False, True])

print(f"--- Auditoría de Equipos en Champions_League (UCL vs UWCL) ---")
print(f"Total de equipos únicos encontrados: {len(df_audit)}")

# 4. Mostrar el DataFrame para tu revisión manual
# Los que tengan 'probable_femenino' True irán arriba.
display(df_audit)

--- Auditoría de Equipos en Champions_League (UCL vs UWCL) ---
Total de equipos únicos encontrados: 329


,Team_id,Team,match_count,probable_femenino
13,092980d3,Apollon Ladies,16,True
198,9f44dae3,WFC Chertanovo,2,True
300,ef289882,1 FC Slovacko,6,False
31,15e031af,AC Omonia,4,False
268,d6a485e1,AC Virtus,1,False
...,...,...,...,...
64,2a6cfcb2,Zrinjski Mostar,1,False
52,22a5a99c,Zurich,1,False
157,82a7cabd,Zvezda Perm,2,False
132,6d12ffd1,Þor KA,2,False


In [43]:
# 1. Extraer el Universo de IDs Masculinos Confirmados
# Tomamos todos los IDs únicos de las ligas nacionales (asumiendo que df_comp_check_limpio ya no tiene femeninos)
ids_masculinos_confirmados = set(df_comp_check_limpio['Team_id'].astype(str).unique())

# 2. Identificar el "Residuo" en Champions League
# Filtramos la Champions y buscamos quiénes NO están en la élite masculina confirmada
df_ucl = df_comp_check[df_comp_check["competition_raw"] == "Champions_League"]

df_residuo = (
    df_ucl[~df_ucl["Team_id"].astype(str).isin(ids_masculinos_confirmados)]
    .groupby(["Team_id", "Team"])
    .size()
    .reset_index(name="match_count")
    .sort_values("match_count", ascending=False)
)

# 3. Reporte Codex
print(f"--- Análisis de Residuo de Champions ---")
print(f"Equipos en UCL: {df_ucl['Team_id'].nunique()}")
print(f"Equipos masculinos confirmados (vía Ligas): {len(ids_masculinos_confirmados.intersection(set(df_ucl['Team_id']))) }")
print(f"Equipos en el 'Residuo' para revisión: {len(df_residuo)}")

# Mostrar los sospechosos de ser femeninos o equipos menores
display(df_residuo)

--- Análisis de Residuo de Champions ---
Equipos en UCL: 326
Equipos masculinos confirmados (vía Ligas): 310
Equipos en el 'Residuo' para revisión: 16


,Team_id,Team,match_count
8,80595417,Paris S-G,49
14,d48ad4ff,Napoli,40
4,3d377a8c,St Polten,37
6,613577b8,Juventus,36
0,02f8d026,AS Roma,25
3,39abd503,Glasgow City,23
7,7213da33,Lazio,16
12,c1cf4fdc,Sparta Prague,16
15,e83b563b,Servette,16
13,c99ff6e5,Fiorentina,14


In [44]:
# Ver todos los valores únicos de competition_raw en el DataFrame limpio
display(df_comp_check_limpio['competition_raw'].value_counts())
print('Valores únicos:', df_comp_check_limpio['competition_raw'].unique())

,count
competition_raw,
us_Major_League_Soccer,7696
eng_Premier_League,6454
es_La_Liga,6416
fr_Ligue_1,6066
mx_Liga_MX,5720
jp_J1_League,5598
de_Bundesliga,5162
sa_Saudi_Pro_League,4186
ir_Persian_Gulf_Pro_League,4056


Valores únicos: ['br_Série_A' 'ir_Persian_Gulf_Pro_League' 'mx_Liga_MX' 'fr_Ligue_1'
 'jp_J1_League' 've_Venezuelan_Primera_Division' 'us_Major_League_Soccer'
 've_Venezuelan_Primera_División' 'sa_Saudi_Pro_League'
 'eng_Premier_League' 'Champions_League' 'Europa_League' 'de_Bundesliga'
 'es_La_Liga' 'gr_Super_League_Greece' 'ar_Superliga_Argentina'
 'Copa_Libertadores' 'Copa_Sudamericana' 'kr_K_League_1'
 'FIFA_Club_World_Cup' 'CONCACAF_Champions_League'
 'ar_Liga_Profesional_Argentina' 'Europa_Conference_League' 'it_Serie_A'
 'CONCACAF_Champions_Cup' 'Conference_League']


In [45]:
# 1. Definir la "Zona Segura" (Competiciones 100% masculinas confirmadas)
zona_segura_comps = [
    'us_Major_League_Soccer', 'eng_Premier_League', 'es_La_Liga', 'mx_Liga_MX',
    'jp_J1_League', 'fr_Ligue_1', 'sa_Saudi_Pro_League', 'ir_Persian_Gulf_Pro_League',
    'de_Bundesliga', 'gr_Super_League_Greece', 'ar_Liga_Profesional_Argentina',
    'br_Série_A', 've_Venezuelan_Primera_División', 'kr_K_League_1',
    'Europa_League', 'Copa_Libertadores', 'Copa_Sudamericana', 'ar_Superliga_Argentina',
    've_Venezuelan_Primera_Division', 'Europa_Conference_League', 'it_Serie_A',
    'Conference_League', 'CONCACAF_Champions_Cup', 'FIFA_Club_World_Cup'
]

# 2. Extraer IDs Masculinos de la Zona Segura
# Filtramos df_comp_check_limpio (o el original si prefieres) buscando estas ligas
ids_masculinos_safe = set(
    df_comp_check[df_comp_check['competition_raw'].isin(zona_segura_comps)]['Team_id'].astype(str).unique()
)

# 3. Filtrar el Residuo de Champions League
# Seleccionamos solo los partidos de Champions donde el Team_id NO está en la zona segura
df_ucl_residuo = (
    df_comp_check[df_comp_check["competition_raw"] == "Champions_League"]
    .pipe(lambda d: d[~d["Team_id"].astype(str).isin(ids_masculinos_safe)])
    .groupby(["Team_id", "Team"])
    .size()
    .reset_index(name="match_count")
    .sort_values("match_count", ascending=False)
)

# 4. Reporte de Auditoría Proactiva
print(f"--- Diagnóstico de Champions (Residual) ---")
print(f"IDs masculinos detectados en ligas seguras: {len(ids_masculinos_safe):,}")
print(f"Equipos en Champions que no están en ligas seguras: {len(df_ucl_residuo)}")

# Mostrar los sospechosos (Aquí estarán los femeninos y los masculinos de ligas no incluidas arriba)
display(df_ucl_residuo)

--- Diagnóstico de Champions (Residual) ---
IDs masculinos detectados en ligas seguras: 922
Equipos en Champions que no están en ligas seguras: 150


,Team_id,Team,match_count
16,15f49df1,Barcelona,63
73,7f2012ad,Lyon,60
102,a1393014,Wolfsburg,54
105,a6a4e67d,Chelsea,51
53,51ec22be,Bayern Munich,49
...,...,...,...
92,931cb21e,Ramat HaSharon,1
108,ae156985,KR Reykjavik,1
123,c306c432,HJK Helsinki,1
129,d6a485e1,AC Virtus,1


In [46]:
import pandas as pd

# 1. Configuración de visualización de Pandas para ver TODO
pd.set_option('display.max_rows', 200) 

# 2. Obtener nombres de equipos masculinos para comparación cruzada
nombres_masculinos_safe = set(df_comp_check[df_comp_check['competition_raw'].isin(zona_segura_comps)]['Team'].unique())

# 3. Construcción de la tabla de correspondencia completa del Residuo
df_ucl_residuo_full = (
    df_ucl_residuo[~df_ucl_residuo["Team_id"].astype(str).isin(ids_masculinos_safe)]
    .groupby(["Team_id", "Team"])
    .size()
    .reset_index(name="match_count")
)

# 4. Auditoría de nombres duplicados (Codex Proactive Logic)
# Si el nombre existe en hombres pero el ID es nuevo, es altamente sospechoso de ser femenino
df_ucl_residuo_full["nombre_en_masculino_safe"] = df_ucl_residuo_full["Team"].isin(nombres_masculinos_safe)

# 5. Ordenar por match_count (los más pesados arriba son los femeninos top)
df_ucl_residuo_full = df_ucl_residuo_full.sort_values("match_count", ascending=False).reset_index(drop=True)

print(f"--- TABLA DE CORRESPONDENCIA COMPLETA: RESIDUO UCL ---")
print(f"Instrucción: Revisa especialmente los que tienen 'nombre_en_masculino_safe' = True")
display(df_ucl_residuo_full)

--- TABLA DE CORRESPONDENCIA COMPLETA: RESIDUO UCL ---
Instrucción: Revisa especialmente los que tienen 'nombre_en_masculino_safe' = True


,Team_id,Team,match_count,nombre_en_masculino_safe
0,005a8517,Breiðablik,1,True
1,0341a929,NSA Sofia,1,False
2,0417d25a,Rangers,1,True
3,04a17aa1,Czarni Sosnowiec,1,False
4,07787bf7,Sturm Graz,1,True
5,092980d3,Apollon Ladies,1,False
6,0a6448a7,FK Sileks,1,False
7,0b92bbb1,Spartak Myjava,1,False
8,0bbd83f6,Manchester Utd,1,True
9,0d19b2b7,Osijek,1,True


In [47]:
# 1. Definir los IDs de Élite (Big 5 + Grecia)
# Filtramos solo las ligas que mencionaste para crear nuestra "Whitelist de Élite"
ligas_elite = [
    'us_Major_League_Soccer', 'eng_Premier_League', 'es_La_Liga', 'mx_Liga_MX',
    'jp_J1_League', 'fr_Ligue_1', 'sa_Saudi_Pro_League', 'ir_Persian_Gulf_Pro_League',
    'de_Bundesliga', 'gr_Super_League_Greece', 'ar_Liga_Profesional_Argentina',
    'br_Série_A', 've_Venezuelan_Primera_División', 'kr_K_League_1', 'ar_Superliga_Argentina',
    've_Venezuelan_Primera_Division', 'it_Serie_A'
]

copas_a_limpiar = [
    'Champions_League', 'Europa_League', 'Conference_League', 
    'Copa_Libertadores', 'Copa_Sudamericana', 
    'CONCACAF_Champions_League', 'CONCACAF_Champions_Cup', 
    'Europa_Conference_League', 'FIFA_Club_World_Cup'
]

ids_elite = set(
    df_comp_check[df_comp_check['competition_raw'].isin(ligas_elite)]['Team_id'].astype(str).unique()
)

# 2. Aplicar el Doble Filtro al DataFrame Principal
# Condición: El partido debe ser de una de las ligas élite 
# O ser de Champions/Europa League PERO con ambos equipos pertenecientes a la élite.

mask_ligas_elite = df_comp_check['competition_raw'].isin(ligas_elite)

mask_copas_elite = (
    df_comp_check['competition_raw'].isin(copas_a_limpiar) & 
    df_comp_check['team_id_home'].astype(str).isin(ids_elite) & 
    df_comp_check['team_id_away'].astype(str).isin(ids_elite)
)

df_final_opta = df_comp_check[mask_ligas_elite | mask_copas_elite].copy()

# 3. Auditoría de Calidad
print(f"--- Resultado de la Purga de Élite ---")
print(f"Partidos originales: {len(df_comp_check):,}")
print(f"Partidos tras doble limpieza: {len(df_final_opta):,}")
print(f"Equipos únicos restantes: {df_final_opta['Team_id'].nunique()}")

--- Resultado de la Purga de Élite ---
Partidos originales: 93,002
Partidos tras doble limpieza: 80,886
Equipos únicos restantes: 442


In [48]:
df_final_opta['competition_raw'].value_counts()
#df_comp_check['competition_raw'].value_counts()

,count
competition_raw,
us_Major_League_Soccer,7696
eng_Premier_League,6454
es_La_Liga,6416
fr_Ligue_1,6066
mx_Liga_MX,5720
jp_J1_League,5598
de_Bundesliga,5162
it_Serie_A,4280
sa_Saudi_Pro_League,4186


In [49]:
# Unificar nombres de competencias
df_final_opta = df_final_opta.copy()
df_final_opta["Competition"] = df_final_opta["competition_raw"]

# Renombrar casos ambiguos/duplicados primero
competition_renames = {
    "br_Série_A": "Brasileirao",
    "ve_Venezuelan_Primera_División": "FutVe",
    "ve_Venezuelan_Primera_Division": "FutVe",
    "Conference_League": "EuropaConferenceLeague"
}
df_final_opta["Competition"] = df_final_opta["Competition"].replace(competition_renames)

# Quitar prefijo de pais para el resto de ligas
df_final_opta["Competition"] = df_final_opta["Competition"].str.replace(
    r"^[a-z]{2,3}_", "", regex=True
 )

# Eliminar guiones bajos para compactar
df_final_opta["Competition"] = df_final_opta["Competition"].str.replace("_", "", regex=False)

df_final_opta["Competition"].value_counts().head(30)

,count
Competition,
MajorLeagueSoccer,7696
PremierLeague,6454
LaLiga,6416
Ligue1,6066
LigaMX,5720
J1League,5598
FutVe,5294
Bundesliga,5162
SerieA,4280


In [50]:
# Recorte temporal por competencias usando match_id_new (YYYYMMDD_hometeamid_awayteamid).
df_final_opta2 = df_final_opta.copy()

# Derivar fecha desde match_id_new para evitar dependencia de columnas de fecha externas.
if "match_id_new" in df_final_opta2.columns:
    date_str = df_final_opta2["match_id_new"].astype(str).str.slice(0, 8)
    df_final_opta2["match_date_from_id"] = pd.to_datetime(
        date_str, format="%Y%m%d", errors="coerce"
    )
else:
    df_final_opta2["match_date_from_id"] = pd.NaT

comp_col = "Competition" if "Competition" in df_final_opta2.columns else "competition_raw"
comp_norm = df_final_opta2[comp_col].astype(str).str.lower()

# Ventana 1: 01-01-2017 a 31-12-2017
comps_2017 = [
    "majorleaguesoccer",
    "j1league",
    "futve",
    "brasileirao",
    "kleague1",
]
start_2017 = pd.to_datetime("01-01-2017", dayfirst=True)
end_2017 = pd.to_datetime("31-12-2017", dayfirst=True)
mask_2017 = comp_norm.isin(comps_2017) & df_final_opta2["match_date_from_id"].between(
    start_2017, end_2017
 )

# Ventana 2: 01-08-2025 a 31-12-2025
comps_2025 = [
    "premierleague",
    "laliga",
    "ligue1",
    "ligamx",
    "bundesliga",
    "seriea",
    "saudiproleague",
    "persiangulfproleague",
    "superleaguegreece",
    "championsleague",
    "europaleague",
    "europaconferenceleague",
]
start_2025 = pd.to_datetime("01-08-2025", dayfirst=True)
end_2025 = pd.to_datetime("31-12-2025", dayfirst=True)
mask_2025 = comp_norm.isin(comps_2025) & df_final_opta2["match_date_from_id"].between(
    start_2025, end_2025
 )

to_drop = mask_2017 | mask_2025
print("Filas antes:", len(df_final_opta2))
print("Filas a eliminar:", int(to_drop.sum()))
df_final_opta2 = df_final_opta2[~to_drop].copy()
print("Filas despues:", len(df_final_opta2))

Filas antes: 80886
Filas a eliminar: 3602
Filas despues: 77284


In [51]:
df_final_opta2.columns.tolist()

['Team_id',
 'Goals',
 'Assists',
 'GoalsPen',
 'AttPen',
 'Shots',
 'ShTarget',
 'YellowCard',
 'RedCard',
 'Fouls',
 'FoulsD',
 'Offside',
 'Crosses',
 'TacklesW',
 'Interceptions',
 'OwnGoals',
 'PenKickGen',
 'PenKickCon',
 'home_team',
 'away_team',
 'Team',
 'match_id',
 'source_file',
 'match_id_new',
 'match_date',
 'team_id_home',
 'team_id_away',
 'source_key',
 'source_key_ascii',
 'competition',
 'competition_raw',
 'competition_raw_norm',
 'Competition',
 'match_date_from_id']

In [52]:
# Separar en dos DB paralelas: local y visita
df_final_home = df_final_opta2[df_final_opta2["Team"] == df_final_opta2["home_team"]].copy()
df_final_away = df_final_opta2[df_final_opta2["Team"] == df_final_opta2["away_team"]].copy()

print("Filas total:", len(df_final_opta2))
print("Filas home:", len(df_final_home))
print("Filas away:", len(df_final_away))

Filas total: 77284
Filas home: 38642
Filas away: 38642


In [53]:
df_final_home.head(5)

,Team_id,Goals,Assists,GoalsPen,AttPen,Shots,ShTarget,YellowCard,RedCard,Fouls,...,match_date,team_id_home,team_id_away,source_key,source_key_ascii,competition,competition_raw,competition_raw_norm,Competition,match_date_from_id
4,0230c3aa,1.0,1.0,0.0,0.0,NaN,NaN,1.0,0.0,NaN,...,03-08-2017,0230c3aa,95f42e44,players_ir_persian_gulf_pro_league_03_08_2017_...,players_ir_persian_gulf_pro_league_03_08_2017_...,Persian_Gulf_Pro_League,ir_Persian_Gulf_Pro_League,ir_persian_gulf_pro_league,PersianGulfProLeague,2017-08-03
8,6b849eeb,3.0,2.0,1.0,1.0,NaN,NaN,4.0,0.0,NaN,...,03-08-2017,6b849eeb,9fc3b195,players_ir_persian_gulf_pro_league_03_08_2017_...,players_ir_persian_gulf_pro_league_03_08_2017_...,Persian_Gulf_Pro_League,ir_Persian_Gulf_Pro_League,ir_persian_gulf_pro_league,PersianGulfProLeague,2017-08-03
11,aa3eb1d3,1.0,1.0,0.0,0.0,NaN,NaN,0.0,0.0,NaN,...,03-08-2017,aa3eb1d3,76d09801,players_ir_persian_gulf_pro_league_03_08_2017_...,players_ir_persian_gulf_pro_league_03_08_2017_...,Persian_Gulf_Pro_League,ir_Persian_Gulf_Pro_League,ir_persian_gulf_pro_league,PersianGulfProLeague,2017-08-03
13,df40b697,1.0,0.0,1.0,1.0,NaN,NaN,2.0,0.0,NaN,...,03-08-2017,df40b697,9c740560,players_ir_persian_gulf_pro_league_03_08_2017_...,players_ir_persian_gulf_pro_league_03_08_2017_...,Persian_Gulf_Pro_League,ir_Persian_Gulf_Pro_League,ir_persian_gulf_pro_league,PersianGulfProLeague,2017-08-03
14,5835aae0,3.0,3.0,0.0,0.0,NaN,NaN,3.0,0.0,NaN,...,04-08-2017,5835aae0,c659a2f9,players_ir_persian_gulf_pro_league_04_08_2017_...,players_ir_persian_gulf_pro_league_04_08_2017_...,Persian_Gulf_Pro_League,ir_Persian_Gulf_Pro_League,ir_persian_gulf_pro_league,PersianGulfProLeague,2017-08-04


In [54]:
# Renombrar metricas deportivas en home y away
metrics_cols = [
    "Team_id", "Goals", "Assists", "GoalsPen", "AttPen", "Shots", "ShTarget", "YellowCard", "RedCard",
    "Fouls", "FoulsD", "Offside", "Crosses", "TacklesW", "Interceptions", "OwnGoals",
    "PenKickGen", "PenKickCon",
]

home_rename = {col: f"home_{col.lower()}" for col in metrics_cols if col in df_final_home.columns}
away_rename = {col: f"away_{col.lower()}" for col in metrics_cols if col in df_final_away.columns}

df_final_home = df_final_home.rename(columns=home_rename)
df_final_away = df_final_away.rename(columns=away_rename)

print("Renombradas home:", sorted(home_rename.values()))
print("Renombradas away:", sorted(away_rename.values()))

Renombradas home: ['home_assists', 'home_attpen', 'home_crosses', 'home_fouls', 'home_foulsd', 'home_goals', 'home_goalspen', 'home_interceptions', 'home_offside', 'home_owngoals', 'home_penkickcon', 'home_penkickgen', 'home_redcard', 'home_shots', 'home_shtarget', 'home_tacklesw', 'home_team_id', 'home_yellowcard']
Renombradas away: ['away_assists', 'away_attpen', 'away_crosses', 'away_fouls', 'away_foulsd', 'away_goals', 'away_goalspen', 'away_interceptions', 'away_offside', 'away_owngoals', 'away_penkickcon', 'away_penkickgen', 'away_redcard', 'away_shots', 'away_shtarget', 'away_tacklesw', 'away_team_id', 'away_yellowcard']


In [55]:
# Unificar home/away en un solo DF final
df_final_cout = df_final_opta2.copy()

base_cols = ["match_id_new", "Competition", "match_date_from_id", "source_key_ascii"]
base_cols = [c for c in base_cols if c in df_final_cout.columns]
df_final_cout = df_final_cout[base_cols].copy()

home_cols = [c for c in df_final_home.columns if c.startswith("home_") or c == "match_id_new"]
away_cols = [c for c in df_final_away.columns if c.startswith("away_") or c == "match_id_new"]

df_home_subset = df_final_home[home_cols].copy()
df_away_subset = df_final_away[away_cols].copy()

df_final_cout = df_final_cout.merge(df_home_subset, on="match_id_new", how="inner")
df_final_cout = df_final_cout.merge(df_away_subset, on="match_id_new", how="inner")

print("Filas base:", len(df_final_opta2))
print("Filas home:", len(df_final_home))
print("Filas away:", len(df_final_away))
print("Filas final:", len(df_final_cout))
df_final_cout.head()

Filas base: 77284
Filas home: 38642
Filas away: 38642
Filas final: 77284


,match_id_new,Competition,match_date_from_id,source_key_ascii,home_team_id,home_goals,home_assists,home_goalspen,home_attpen,home_shots,...,away_fouls,away_foulsd,away_offside,away_crosses,away_tacklesw,away_interceptions,away_owngoals,away_penkickgen,away_penkickcon,away_team
0,20170803_0230c3aa_95f42e44,PersianGulfProLeague,2017-08-03,players_ir_persian_gulf_pro_league_03_08_2017_...,0230c3aa,1.0,1.0,0.0,0.0,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,0.0,NaN,NaN,Persepolis
1,20170803_0230c3aa_95f42e44,PersianGulfProLeague,2017-08-03,players_ir_persian_gulf_pro_league_03_08_2017_...,0230c3aa,1.0,1.0,0.0,0.0,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,0.0,NaN,NaN,Persepolis
2,20170803_6b849eeb_9fc3b195,PersianGulfProLeague,2017-08-03,players_ir_persian_gulf_pro_league_03_08_2017_...,6b849eeb,3.0,2.0,1.0,1.0,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,0.0,NaN,NaN,Sanat Naft Abadan
3,20170803_6b849eeb_9fc3b195,PersianGulfProLeague,2017-08-03,players_ir_persian_gulf_pro_league_03_08_2017_...,6b849eeb,3.0,2.0,1.0,1.0,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,0.0,NaN,NaN,Sanat Naft Abadan
4,20170803_aa3eb1d3_76d09801,PersianGulfProLeague,2017-08-03,players_ir_persian_gulf_pro_league_03_08_2017_...,aa3eb1d3,1.0,1.0,0.0,0.0,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,0.0,NaN,NaN,Naft Tehran


In [56]:
# Diagnostico de duplicados por match_id_new
dup_home = df_final_home["match_id_new"].value_counts()
dup_away = df_final_away["match_id_new"].value_counts()
dup_base = df_final_opta2["match_id_new"].value_counts()

print("match_id_new duplicados en home:", int((dup_home > 1).sum()))
print("match_id_new duplicados en away:", int((dup_away > 1).sum()))
print("match_id_new duplicados en base:", int((dup_base > 2).sum()))

dup_home_ids = dup_home[dup_home > 1].index.tolist()
dup_away_ids = dup_away[dup_away > 1].index.tolist()
dup_base_ids = dup_base[dup_base > 2].index.tolist()

print("Ejemplos home:", dup_home_ids[:10])
print("Ejemplos away:", dup_away_ids[:10])
print("Ejemplos base:", dup_base_ids[:10])

match_id_new duplicados en home: 0
match_id_new duplicados en away: 0
match_id_new duplicados en base: 0
Ejemplos home: []
Ejemplos away: []
Ejemplos base: []


In [57]:
# Conteo detallado de duplicados por match_id_new
dup_home_counts = dup_home[dup_home > 1].reset_index()
dup_home_counts.columns = ["match_id_new", "count_home"]

dup_away_counts = dup_away[dup_away > 1].reset_index()
dup_away_counts.columns = ["match_id_new", "count_away"]

dup_base_counts = dup_base[dup_base > 2].reset_index()
dup_base_counts.columns = ["match_id_new", "count_base"]

print("Duplicados home (top 20):")
display(dup_home_counts.head(20))
print("Duplicados away (top 20):")
display(dup_away_counts.head(20))
print("Duplicados base (top 20):")
display(dup_base_counts.head(20))

Duplicados home (top 20):


,match_id_new,count_home


Duplicados away (top 20):


,match_id_new,count_away


Duplicados base (top 20):


,match_id_new,count_base


In [58]:
# Mostrar filas completas de los duplicados
df_dup_home_rows = df_final_home[df_final_home["match_id_new"].isin(dup_home_ids)].copy()
df_dup_away_rows = df_final_away[df_final_away["match_id_new"].isin(dup_away_ids)].copy()
df_dup_base_rows = df_final_opta2[df_final_opta2["match_id_new"].isin(dup_base_ids)].copy()

df_dup_home_rows = df_dup_home_rows.sort_values(["match_id_new"])
df_dup_away_rows = df_dup_away_rows.sort_values(["match_id_new"])
df_dup_base_rows = df_dup_base_rows.sort_values(["match_id_new"])

print("Filas duplicadas (home):", len(df_dup_home_rows))
display(df_dup_home_rows)
print("Filas duplicadas (away):", len(df_dup_away_rows))
display(df_dup_away_rows)
print("Filas duplicadas (base):", len(df_dup_base_rows))
display(df_dup_base_rows)

Filas duplicadas (home): 0


,home_team_id,home_goals,home_assists,home_goalspen,home_attpen,home_shots,home_shtarget,home_yellowcard,home_redcard,home_fouls,...,match_date,team_id_home,team_id_away,source_key,source_key_ascii,competition,competition_raw,competition_raw_norm,Competition,match_date_from_id


Filas duplicadas (away): 0


,away_team_id,away_goals,away_assists,away_goalspen,away_attpen,away_shots,away_shtarget,away_yellowcard,away_redcard,away_fouls,...,match_date,team_id_home,team_id_away,source_key,source_key_ascii,competition,competition_raw,competition_raw_norm,Competition,match_date_from_id


Filas duplicadas (base): 0


,Team_id,Goals,Assists,GoalsPen,AttPen,Shots,ShTarget,YellowCard,RedCard,Fouls,...,match_date,team_id_home,team_id_away,source_key,source_key_ascii,competition,competition_raw,competition_raw_norm,Competition,match_date_from_id


In [59]:
# Rastreo completo de Team_id especifico
target_team_id = "990519b8"

df_team_all = df_final_opta2[df_final_opta2["Team_id"].astype(str) == target_team_id].copy()

print("Filas encontradas:", len(df_team_all))
print("Equipos (Team) unicos:", sorted(df_team_all["Team"].dropna().unique()))
print("Home_team unicos:", sorted(df_team_all["home_team"].dropna().unique()))
print("Away_team unicos:", sorted(df_team_all["away_team"].dropna().unique()))
print("Competitions unicas:", sorted(df_team_all["Competition"].dropna().unique()) if "Competition" in df_team_all.columns else [])

display(df_team_all.sort_values(["match_id_new"]))

Filas encontradas: 0
Equipos (Team) unicos: []
Home_team unicos: []
Away_team unicos: []
Competitions unicas: []


,Team_id,Goals,Assists,GoalsPen,AttPen,Shots,ShTarget,YellowCard,RedCard,Fouls,...,match_date,team_id_home,team_id_away,source_key,source_key_ascii,competition,competition_raw,competition_raw_norm,Competition,match_date_from_id


In [60]:
# Listado de source_key_ascii para el Team_id objetivo
if "source_key_ascii" in df_team_all.columns:
    source_keys = sorted(df_team_all["source_key_ascii"].dropna().unique().tolist())
else:
    source_keys = []

print("Total source_key_ascii:", len(source_keys))
print(source_keys)

Total source_key_ascii: 0
[]


In [61]:
# Drop del Team_id objetivo en df_final_opta2, df_final_home y df_final_away
drop_team_id = "990519b8"

def _drop_team_id(df):
    df = df.copy()
    mask = pd.Series(False, index=df.index)
    if "match_id_new" in df.columns:
        mask = mask | df["match_id_new"].astype(str).str.contains(drop_team_id, na=False)
    # Usar nombres reales de columnas si existen
    for col in ["team_id_home", "team_id_away", "home_team_id", "away_team_id"]:
        if col in df.columns:
            mask = mask | (df[col].astype(str) == drop_team_id)
    # Si hay columna Team_id directa, tambien se considera
    if "Team_id" in df.columns:
        mask = mask | (df["Team_id"].astype(str) == drop_team_id)
    return df[~mask].copy()

df_final_opta2 = _drop_team_id(df_final_opta2)
df_final_home = _drop_team_id(df_final_home)
df_final_away = _drop_team_id(df_final_away)

print("Filas despues (base):", len(df_final_opta2))
print("Filas despues (home):", len(df_final_home))
print("Filas despues (away):", len(df_final_away))

Filas despues (base): 77284
Filas despues (home): 38642
Filas despues (away): 38642


In [62]:
# Cargar bases externas para variables adicionales
#path_estadiums = r"C:\Users\ruben\Desktop\TEG\EstadiumsDB.csv"
#path_macro = r"C:\Users\ruben\Desktop\TEG\MacroEconomy2_imputado.csv"
#path_teams_econ = r"C:\Users\ruben\Desktop\TEG\TeamsEconomyDB.csv"

path_estadiums = os.path.join(BASE_DATA_DIR, "EstadiumsDB.csv")
path_macro = os.path.join(BASE_DATA_DIR, "MacroEconomy2_imputado.csv")
path_teams_econ = os.path.join(BASE_DATA_DIR, "TeamsEconomyDB.csv")

df_estadiums = pd.read_csv(path_estadiums)
df_macro = pd.read_csv(path_macro)
df_teams_econ = pd.read_csv(path_teams_econ)

print("Estadiums:", df_estadiums.shape)
print("Macro:", df_macro.shape)
print("TeamsEconomy:", df_teams_econ.shape)

Estadiums: (437, 15)
Macro: (135, 9)
TeamsEconomy: (6276, 9)


In [63]:
# Insertar columna Country en home/away (se completara despues)
if "Country" not in df_final_home.columns:
    df_final_home.insert(len(df_final_home.columns), "Country", pd.NA)
if "Country" not in df_final_away.columns:
    df_final_away.insert(len(df_final_away.columns), "Country", pd.NA)

print("Country agregado en home:", "Country" in df_final_home.columns)
print("Country agregado en away:", "Country" in df_final_away.columns)

Country agregado en home: True
Country agregado en away: True


In [64]:
# Listas manuales de nombres (Team y EstadiumsDB)
def _pick_col(df, preferred):
    for col in preferred:
        if col in df.columns:
            return col
    return None

# Lista de Team (home + away)
team_col = "Team" if "Team" in df_final_home.columns else None
if team_col is None:
    print("No se encontro columna Team en df_final_home.")
else:
    team_names = pd.concat([
        df_final_home[team_col],
        df_final_away[team_col] if team_col in df_final_away.columns else pd.Series([], dtype=object),
    ], ignore_index=True)
    team_names = sorted(team_names.dropna().astype(str).unique().tolist())
    print("Total Team unicos:", len(team_names))
    print(team_names)

# Lista de nombres desde EstadiumsDB (elige la mejor columna disponible)
est_name_col = _pick_col(
    df_estadiums,
    ["Team", "team", "Club", "club", "Team_name", "team_name", "Club_name", "club_name", "Stadium", "stadium", "Name", "name"],
 )
if est_name_col is None:
    print("No se encontro columna de nombre en df_estadiums. Columnas disponibles:")
    print(sorted(df_estadiums.columns.tolist()))
else:
    est_names = sorted(df_estadiums[est_name_col].dropna().astype(str).unique().tolist())
    print(f"Columna usada en EstadiumsDB: {est_name_col}")
    print("Total nombres unicos en EstadiumsDB:", len(est_names))
    print(est_names)

Total Team unicos: 431
['AEK Athens', 'AO Kifisia', 'AS Roma', 'ASD Calcio Pomigliano', 'Abha', 'Ajaccio', 'Al Kholood Saudi Club', 'Al Okhdood SC', 'Al-Adalah', 'Al-Ahli', 'Al-Ain', 'Al-Batin', 'Al-Ettifaq', 'Al-Faisaly', 'Al-Fateh', 'Al-Fayha', 'Al-Hazem', 'Al-Hilal', 'Al-Ittihad', 'Al-Khaleej', 'Al-Nassr', 'Al-Orobah', 'Al-Qadsiah', 'Al-Raed', 'Al-Riyadh SC', 'Al-Shabab', "Al-Ta'ee", 'Al-Taawoun', 'Al-Wehda', 'Alaves', 'Albirex Niigata', 'Aldosivi', 'Almeria', 'Aluminium Arak', 'America', 'America–MG', 'Amiens', 'Angers', 'Angostura', 'Anzoategui', 'Apollon Smyrnis', 'Aragua', 'Arg Juniors', 'Aris', 'Arminia', 'Arsenal', 'Asteras Tripoli', 'Aston Villa', 'Atalanta', 'Athletic Club', 'Athletico Paranaense', 'Atlanta Utd', 'Atlas', 'Atle Tucuman', 'Atletico Goianiense', 'Atletico Madrid', 'Atletico Mineiro', 'Atletico San Luis', 'Atletico Venezuela', 'Atromitos', 'Augsburg', 'Austin FC', 'Auxerre', 'Avai', 'Avispa Fukuoka', 'Bahia', 'Banfield', 'Barcelona', 'Barracas Central', 'Bayern

In [65]:
# Buscar equipos femeninos previamente limpiados y mostrar Team + Competition
if "women_teams" in globals():
    target_women = [str(x) for x in women_teams]
else:
    target_women = []

df_comp_src = df_final_opta2.copy() if "df_final_opta2" in globals() else df_final_opta.copy()
cols_show = [c for c in ["Team", "Team_id", "Competition"] if c in df_comp_src.columns]
if target_women and "Team" in df_comp_src.columns:
    df_women_found = df_comp_src[df_comp_src["Team"].astype(str).isin(target_women)].copy()
else:
    df_women_found = df_comp_src.iloc[0:0].copy()

print("Equipos femeninos detectados (filas):", len(df_women_found))
if cols_show:
    display(df_women_found[cols_show].drop_duplicates().sort_values(["Team", "Competition"]))
else:
    print("Columnas disponibles:", sorted(df_comp_src.columns.tolist()))

# Tabla completa de Team + Competition (para revision manual)
if "Team" in df_comp_src.columns and "Competition" in df_comp_src.columns:
    df_team_comp = (
        df_comp_src[["Team", "Competition"]]
        .dropna()
        .drop_duplicates()
        .sort_values(["Competition", "Team"])
        .reset_index(drop=True)
    )
    print("Total combinaciones Team+Competition:", len(df_team_comp))
    display(df_team_comp)
else:
    print("Falta Team o Competition para construir la tabla.")

Equipos femeninos detectados (filas): 240


,Team,Team_id,Competition
42254,AS Roma,02f8d026,SerieA
42204,ASD Calcio Pomigliano,9f18ced7,SerieA
52391,FC Como Women,756c9acb,SerieA


Total combinaciones Team+Competition: 749


,Team,Competition
0,America–MG,Brasileirao
1,Athletico Paranaense,Brasileirao
2,Atletico Goianiense,Brasileirao
3,Atletico Mineiro,Brasileirao
4,Avai,Brasileirao
...,...,...
744,Talleres–C,SuperligaArgentina
745,Temperley,SuperligaArgentina
746,Tigre,SuperligaArgentina
747,Union,SuperligaArgentina


In [66]:
# Drop de equipos femeninos italianos por Team_id en match_id_new y columnas de ids
#female_it_team_ids = [
    #"02f8d026", "9f18ced7", "a3d88bd8", "756c9acb", "c99ff6e5", "658bf2de",
    #"d35ed67b", "06d05f19", "e0652b02", "7213da33", "dc56fe14", "d48ad4ff",
    #"f7d05cb5", "58e367c3", "e2befd26", "eb8e3e81",
 #]
female_it_team_ids = [
    '02f8d026',   # Roma fem
        '06d05f19',   # Internazionale fem
        '20f5bd86',   # San Marino fem
        '28d9b675',   # Paris Fem
        '331a33a7',   # Tavagnacco fem
        '39abd503',   # Glasgowfem
        '3d377a8c',   # St Polten fem
        '54da9a57',   # Milan Fem
        '658bf2de',   # Genoa Fem
        '19db4c32',   # Lazio fem
        '756c9acb',   # Como Women
        '80595417',   # PSGFem
        '9f18ced7',   # Pomigliano
        'adba8c92',   # Orobica
        'be35aa4d',   # Pink Bari
        'c1cf4fdc',   # Sparta Fem
        'cc6ecd46',   # Chievo fem
        'd42d8b2f',   # Fiorentina fem
        '828ce826',   # Napoli Fem
        'd59e6510',   # AGSMVerona
        'e83b563b',   # Servette fem
        'eb8e3e81',   # Ternana
        'b72bc283',   # Empoli Fem
        '613577b8',   # Juventus Fem 
        'a8e83f23',   # Sassuolo Fem 
        'f7d05cb5',   # Parma Fem 
        'd35ed67b',   # Hellas Verona fem
        '58e367c3',   # Sampdoria
        'a8e83f23',   # Sassuolo
 ]
female_it_set = set(female_it_team_ids)

def _drop_female_it_ids(df):
    df = df.copy()
    mask = pd.Series(False, index=df.index)
    if "match_id_new" in df.columns:
        # match_id_new contiene los ids, construir regex para buscarlos
        pattern = "|".join(female_it_team_ids)
        mask = mask | df["match_id_new"].astype(str).str.contains(pattern, na=False)
    for col in ["team_id_home", "team_id_away", "home_team_id", "away_team_id", "Team_id"]:
        if col in df.columns:
            mask = mask | df[col].astype(str).isin(female_it_set)
    return df[~mask].copy()

print("Filas antes (base):", len(df_final_opta2))
print("Filas antes (home):", len(df_final_home))
print("Filas antes (away):", len(df_final_away))

#df_final_opta2 = _drop_female_it_ids(df_final_opta2)
#df_final_home = _drop_female_it_ids(df_final_home)
#df_final_away = _drop_female_it_ids(df_final_away)

print("Filas despues (base):", len(df_final_opta2))
print("Filas despues (home):", len(df_final_home))
print("Filas despues (away):", len(df_final_away))

Filas antes (base): 77284
Filas antes (home): 38642
Filas antes (away): 38642
Filas despues (base): 77284
Filas despues (home): 38642
Filas despues (away): 38642


In [67]:
# Auditoria: ChampionsLeague con <10 partidos y fuera de ligas elite
df_audit_src = df_final_opta2.copy() if "df_final_opta2" in globals() else df_final_opta.copy()

comp_col = "Competition" if "Competition" in df_audit_src.columns else "competition_raw"
team_id_col = "Team_id" if "Team_id" in df_audit_src.columns else None

elite_comps = {
    "LaLiga", "Ligue1", "PremierLeague", "Bundesliga", "SerieA", "SuperLeagueGreece",
}
champions_comp = "ChampionsLeague"

if comp_col and team_id_col:
    # Team_id de ligas elite
    elite_team_ids = set(
        df_audit_src[df_audit_src[comp_col].isin(elite_comps)][team_id_col]
        .dropna().astype(str).unique().tolist()
    )
    # Filtrar ChampionsLeague fuera de elite
    df_ucl = df_audit_src[df_audit_src[comp_col] == champions_comp].copy()
    df_ucl_non_elite = df_ucl[~df_ucl[team_id_col].astype(str).isin(elite_team_ids)].copy()
    # Contar partidos por Team_id usando match_id_new
    if "match_id_new" in df_ucl_non_elite.columns:
        match_counts = (
            df_ucl_non_elite.groupby([team_id_col, "Team"], dropna=True)["match_id_new"]
            .nunique().reset_index(name="match_count")
        )
    else:
        match_counts = (
            df_ucl_non_elite.groupby([team_id_col, "Team"], dropna=True)
            .size().reset_index(name="match_count")
        )
    audit_small = match_counts[match_counts["match_count"] < 40].sort_values(
        ["match_count", "Team"], ascending=[True, True]
    )
    print("Equipos UCL fuera de ligas elite:", len(match_counts))
    print("Equipos UCL fuera de ligas elite con <40 partidos:", len(audit_small))
    display(audit_small)
else:
    print("Faltan columnas Competition/Team_id para la auditoria.")

Equipos UCL fuera de ligas elite: 0
Equipos UCL fuera de ligas elite con <40 partidos: 0


,Team_id,Team,match_count


In [68]:
# Source_key (top 3) o competiciones para Team_id listados
target_team_ids = [
    "97b7c0e8", "69dee2b0", "0f4738a8", "c21f25d3", "87705c62", "b56c2667",
    "54582b93", "9ce68f8a", "51ec22be", "411b1108", "a1393014", "7f2012ad",
    "a6a4e67d", "15f49df1",
 ]
df_src = df_final_opta2.copy() if "df_final_opta2" in globals() else df_final_opta.copy()

# Asegurar source_key_ascii si existe source_file
if "source_key_ascii" not in df_src.columns and "source_file" in df_src.columns:
    def _source_key_ascii(name):
        if pd.isna(name):
            return name
        base = unicodedata.normalize("NFKD", str(name))
        base = "".join(ch for ch in base if not unicodedata.combining(ch))
        base = base.lower()
        base = re.sub(r"[^a-z0-9]+", "_", base)
        base = re.sub(r"_+", "_", base).strip("_")
        return base
    df_src = df_src.copy()
    df_src["source_key_ascii"] = df_src["source_file"].apply(_source_key_ascii)

cols_needed = [c for c in ["Team_id", "Team", "Competition", "source_key_ascii"] if c in df_src.columns]
df_targets = df_src[df_src["Team_id"].astype(str).isin(target_team_ids)].copy()
if df_targets.empty:
    print("No se encontraron filas para esos Team_id en el dataset actual.")
else:
    # 3 source_key por Team_id
    if "source_key_ascii" in df_targets.columns:
        df_sk = (
            df_targets[["Team_id", "Team", "source_key_ascii"]]
            .dropna()
            .drop_duplicates()
            .groupby(["Team_id", "Team"], dropna=True)["source_key_ascii"]
            .apply(lambda s: sorted(s.unique())[:3])
            .reset_index(name="source_key_top3")
        )
        display(df_sk)
    # Competiciones por Team_id
    if "Competition" in df_targets.columns:
        df_comp = (
            df_targets[["Team_id", "Team", "Competition"]]
            .dropna()
            .drop_duplicates()
            .groupby(["Team_id", "Team"], dropna=True)["Competition"]
            .apply(lambda s: sorted(s.unique()))
            .reset_index(name="competitions")
        )
        display(df_comp)

No se encontraron filas para esos Team_id en el dataset actual.


In [69]:
# Buscar equipos especificos (posibles femeninos) y mostrar detalles
search_terms = [
    "Apollon Ladies",
    "Osijek",
    "Swansea City",
    "Aland United",
    "GKS GieKSa Katowice",
 ]

df_search_src = df_final_opta2.copy() if "df_final_opta2" in globals() else df_final_opta.copy()
cols_out = [c for c in ["Team", "Team_id", "Competition", "source_key_ascii", "match_id_new"] if c in df_search_src.columns]

mask_terms = df_search_src["Team"].astype(str).str.contains(
    "|".join(search_terms), case=False, na=False, regex=True
)
df_hits = df_search_src[mask_terms].copy()
print("Filas encontradas:", len(df_hits))
if cols_out:
    display(
        df_hits[cols_out]
        .drop_duplicates()
        .sort_values(["Team", "Competition"], ascending=[True, True])
    )
else:
    print("Columnas disponibles:", sorted(df_search_src.columns.tolist()))

Filas encontradas: 38


,Team,Team_id,Competition,source_key_ascii,match_id_new
195,Swansea City,fb10988f,PremierLeague,players_eng_premier_league_12_08_2017_southamp...,20170812_33c895d4_fb10988f
509,Swansea City,fb10988f,PremierLeague,players_eng_premier_league_19_08_2017_swansea_...,20170819_fb10988f_19538871
769,Swansea City,fb10988f,PremierLeague,players_eng_premier_league_26_08_2017_crystal_...,20170826_47c64c55_fb10988f
1173,Swansea City,fb10988f,PremierLeague,players_eng_premier_league_10_09_2017_swansea_...,20170910_fb10988f_b2b47a98
1325,Swansea City,fb10988f,PremierLeague,players_eng_premier_league_16_09_2017_tottenha...,20170916_361ca564_fb10988f
1685,Swansea City,fb10988f,PremierLeague,players_eng_premier_league_23_09_2017_swansea_...,20170923_fb10988f_2abfe087
1955,Swansea City,fb10988f,PremierLeague,players_eng_premier_league_30_09_2017_west_ham...,20170930_7c21e445_fb10988f
2271,Swansea City,fb10988f,PremierLeague,players_eng_premier_league_14_10_2017_swansea_...,20171014_fb10988f_f5922ca5
2585,Swansea City,fb10988f,PremierLeague,players_eng_premier_league_21_10_2017_swansea_...,20171021_fb10988f_a2d435b3
2719,Swansea City,fb10988f,PremierLeague,players_eng_premier_league_28_10_2017_arsenal_...,20171028_18bb7c10_fb10988f


In [70]:
# Verificar Team_id listados: cuales aparecen en el dataset actual
target_team_ids_full = [
    "80595417", "81541844", "005a8517", "0341a929", "0417d25a", "04a17aa1", "07787bf7",
 "092980d3", "0a6448a7", "0b92bbb1", "0bbd83f6", "0d19b2b7", "0d683475", "0f31d5f3",
 "0f4738a8", "10207b7a", "10c6ba21", "15f49df1", "167233d5", "1aec6388", "1b3af73b", "1c934f04",
 "1da16b8f", "1e209cf8", "1e615f2b", "215d411b", "231a3620", "236e3619", "23b52962", "23b8ffcd",
 "261c5544", "264d7848", "26a84b38", "28d9b675", "2972399d", "29ee35e0", "2aa6d10a", "2c76cc8f",
 "33fce591", "37b835b5", "3997bac6", "39abd503", "3c94e2df", "3d377a8c", "411b1108", "424e506f",
 "4549ef76", "4573a712", "476d5fea", "4b23c2af", "4f09a762", "4f782ba7", "50fbb58d", "51ec22be",
 "52e4e8e8", "54582b93", "571393d8", "58ff3cd4", "5a12b71d", "5ad3e6fc", "621928ce", "646b9825",
 "6962939c", "69dee2b0", "6d12ffd1", "6e3e89b0", "6f6ee141", "7660add0", "768fae36", "7a1000e8",
 "7ae0d809", "7b538d76", "7ba81fac", "7f2012ad", "814f26cd", "8266975e", "82a7cabd", "82b25b91",
 "8318c196", "856cc63a", "874efae9", "87705c62", "889b0f75", "89f584e1", "8b4cbfb9", "8cf14206",
 "8e9d9e5f", "8ea63662", "92a6e903", "931cb21e", "9522e7b4", "969118cb", "97b7c0e8", "9a505508",
 "9bb79f59", "9ce68f8a", "9d12423d", "9ec14616", "9f44dae3", "a1393014", "a2c4c17a", "a5e92def",
 "a6a4e67d", "a873ed48", "a88780ed", "ae156985", "ae395858", "aee32dcc", "b00041f6", "b16eac73",
 "b56c2667", "b6199a89", "b6f8790a", "b80262ea", "bd5e1428", "beda3b78", "c07cf5b5", "c1c51bdf",
 "c1cf4fdc", "c21f25d3", "c306c432", "c6bd5ba1", "c7efe450", "c843f52a", "ca6492f2", "d2e7fc56",
 "d6a485e1", "d8deaf72", "d9a75109", "da27edc2", "df57dd52", "dfe62afe", "e04b1b89", "e065a5a4",
 "e83b563b", "ea339fbc", "ef289882", "f0dbdec6", "f1e85b1e", "f1f7051d", "f30d7206", "f7dfd35b",
 "f820669e", "f8adab89", "fd1a6b48", "fd573be7", "feb66a71",
]
df_check_src = df_final_opta2.copy() if "df_final_opta2" in globals() else df_final_opta.copy()

if "Team_id" not in df_check_src.columns:
    print("No existe columna Team_id en el dataset actual.")
else:
    df_hits_ids = df_check_src[df_check_src["Team_id"].astype(str).isin(target_team_ids_full)].copy()
    print("Team_id listados encontrados (filas):", len(df_hits_ids))
    df_ids_present = (
        df_hits_ids[["Team_id", "Team", "Competition"]]
        .dropna()
        .drop_duplicates()
        .sort_values(["Team", "Competition"], ascending=[True, True])
    )
    display(df_ids_present)
    ids_present = sorted(df_ids_present["Team_id"].astype(str).unique().tolist())
    ids_missing = sorted(set(target_team_ids_full) - set(ids_present))
    print("Team_id presentes:", len(ids_present))
    print("Team_id ausentes:", len(ids_missing))
    print("Ausentes:", ids_missing)

Team_id listados encontrados (filas): 0


,Team_id,Team,Competition


Team_id presentes: 0
Team_id ausentes: 147
Ausentes: ['005a8517', '0341a929', '0417d25a', '04a17aa1', '07787bf7', '092980d3', '0a6448a7', '0b92bbb1', '0bbd83f6', '0d19b2b7', '0d683475', '0f31d5f3', '0f4738a8', '10207b7a', '10c6ba21', '15f49df1', '167233d5', '1aec6388', '1b3af73b', '1c934f04', '1da16b8f', '1e209cf8', '1e615f2b', '215d411b', '231a3620', '236e3619', '23b52962', '23b8ffcd', '261c5544', '264d7848', '26a84b38', '28d9b675', '2972399d', '29ee35e0', '2aa6d10a', '2c76cc8f', '33fce591', '37b835b5', '3997bac6', '39abd503', '3c94e2df', '3d377a8c', '411b1108', '424e506f', '4549ef76', '4573a712', '476d5fea', '4b23c2af', '4f09a762', '4f782ba7', '50fbb58d', '51ec22be', '52e4e8e8', '54582b93', '571393d8', '58ff3cd4', '5a12b71d', '5ad3e6fc', '621928ce', '646b9825', '6962939c', '69dee2b0', '6d12ffd1', '6e3e89b0', '6f6ee141', '7660add0', '768fae36', '7a1000e8', '7ae0d809', '7b538d76', '7ba81fac', '7f2012ad', '80595417', '814f26cd', '81541844', '8266975e', '82a7cabd', '82b25b91', '8318c196'

In [71]:
# Drop de los 14 Team_id encontrados (protocolo completo por match_id_new y columnas de ids)
drop_ucl_team_ids = [
    "411b1108", "b56c2667", "15f49df1", "51ec22be", "69dee2b0", "a6a4e67d",
    "97b7c0e8", "87705c62", "0f4738a8", "7f2012ad", "9ce68f8a", "54582b93",
    "c21f25d3", "a1393014",
]
drop_ucl_set = set(drop_ucl_team_ids)

def _drop_ucl_ids(df):
    df = df.copy()
    mask = pd.Series(False, index=df.index)
    if "match_id_new" in df.columns:
        pattern = "|".join(drop_ucl_team_ids)
        mask = mask | df["match_id_new"].astype(str).str.contains(pattern, na=False)
    for col in ["team_id_home", "team_id_away", "home_team_id", "away_team_id", "Team_id"]:
        if col in df.columns:
            mask = mask | df[col].astype(str).isin(drop_ucl_set)
    return df[~mask].copy()

print("Filas antes (base):", len(df_final_opta2))
print("Filas antes (home):", len(df_final_home))
print("Filas antes (away):", len(df_final_away))

#df_final_opta2 = _drop_ucl_ids(df_final_opta2)
#df_final_home = _drop_ucl_ids(df_final_home)
#df_final_away = _drop_ucl_ids(df_final_away)

print("Filas despues (base):", len(df_final_opta2))
print("Filas despues (home):", len(df_final_home))
print("Filas despues (away):", len(df_final_away))

Filas antes (base): 77284
Filas antes (home): 38642
Filas antes (away): 38642
Filas despues (base): 77284
Filas despues (home): 38642
Filas despues (away): 38642


In [72]:
# Export final dataframes to CSV
output_dir = Path.cwd()
path_matches = output_dir / "db_matches_final.csv"
path_home = output_dir / "db_matches_final_home.csv"
path_away = output_dir / "db_matches_final_away.csv"

df_final_opta2.to_csv(path_matches, index=False)
df_final_home.to_csv(path_home, index=False)
df_final_away.to_csv(path_away, index=False)

print("Exportado:", path_matches)
print("Exportado:", path_home)
print("Exportado:", path_away)

Exportado: /content/db_matches_final.csv
Exportado: /content/db_matches_final_home.csv
Exportado: /content/db_matches_final_away.csv
